# Sesión 5 — MUBD

**Clustering no supervisado: de la distancia a la interpretación**

**Pregunta de partida:** ¿qué agrupaciones aparecen al comparar observaciones **sin utilizar etiquetas de clase**?

En S2 aprendimos a describir datos y distinguir estimaciones de conclusiones; en S3 separamos ajuste y evaluación. Aquí no hay una respuesta que predecir: elegimos variables, escala y proximidad para explorar grupos.

**Recorrido:** Iris y escala → jerarquía y cortes → Hopkins → vinos → k-means → medoides → geometrías no convexas → atípicos → clientes de un mayorista → síntesis.

**Cómo trabajar:** lee los pasos, ejecuta el ejemplo comentado, interpreta su salida y prueba la micropráctica. Los huecos `...` reutilizan patrones ya mostrados; están aislados del recorrido principal. Las soluciones de los ejercicios 5-1 y 5-2 permanecen completas.

**Puente con la presentación:**
- Introducción y ausencia de respuesta (3–5) → datos sin `Species`.
- Distancias, enlace, Ward e inercia (8–18) → jerarquía.
- Hopkins (19) → tendencia al agrupamiento, no elección de $k$.
- K-means y variantes (23–34) → ajustes con semillas.
- Evaluación interna/externa (35–37) → índices y comparación de particiones.
- Métodos combinados (38–39) → componentes principales y consolidación.

**Regla del análisis:** `Species` se reserva para comparar **después** de agrupar. No se utiliza para entrenar, escoger $k$ ni seleccionar parámetros. Un clúster no es automáticamente una especie ni una predicción supervisada. WSS, variabilidad explicada y porcentajes de PCA **no son porcentajes de aciertos**.

Selecciona el kernel del proyecto y ejecuta en orden con **Shift + Enter**. Usa **Variables** y **Esquema / Outline** para orientarte. Los widgets complementan las llamadas fijas; también se indican alternativas estáticas. Para repetir desde cero, usa **Reiniciar kernel → Ejecutar todo**. [Guía de notebooks en VS Code](https://code.visualstudio.com/docs/datascience/jupyter-notebooks).

## 5.0. Preparación

1. Selecciona el kernel del entorno del proyecto; no es necesario reinstalar las bibliotecas en cada sesión.
2. Reinicia el kernel desde VS Code si necesitas empezar desde cero. El antiguo `%reset -f` se sustituye por un recordatorio: borraría también variables de una comprobación en curso.
3. Ejecuta las importaciones y las funciones de apoyo incluidas aquí. Las funciones largas se comentan **por bloques**; céntrate primero en sus entradas, sus salidas y las llamadas de ejemplo.

Conservamos la semilla general **12345** y la semilla **1234** del ejercicio de comparación. `threadpool_limits` limita a un hilo los ajustes más costosos; no cambia su objetivo estadístico.

In [ ]:
from pathlib import Path  # Rutas al proyecto y a los scripts auxiliares.
import sys

PROYECTO = next((carpeta for carpeta in (Path.cwd(), *Path.cwd().parents)
                 if (carpeta / 'datos' / 'wine2.txt').is_file()), None)
if PROYECTO is None:
    raise FileNotFoundError('Abre el cuaderno dentro del proyecto LaSalle_Estadistica.')
SCRIPTS_PATH = PROYECTO / 'sesiones' / 'sesion5'
if str(SCRIPTS_PATH) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_PATH))  # Funciona desde la raíz o la carpeta de sesión.
DATA_PATH = PROYECTO / 'datos'

In [ ]:
# %pip install numpy pandas matplotlib scipy scikit-learn seaborn ipywidgets threadpoolctl ipykernel

In [ ]:
from functools import partial  # Fija los datos de una función y conserva llamadas cortas.
from types import SimpleNamespace  # Reúne resultados de las funciones de apoyo.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets  # Controles de los experimentos interactivos.
from IPython.display import display
from matplotlib.patches import Circle, Rectangle, Wedge
from pandas.plotting import scatter_matrix
from scipy import stats
from scipy.cluster.hierarchy import linkage, cophenet, cut_tree, dendrogram
from scipy.linalg import eigh
from scipy.spatial.distance import pdist, squareform, cdist
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from threadpoolctl import threadpool_limits
from s5_controles import cabecera_widget, fila_controles
from s5_graficos import (
    configurar_estilo, estilo_ejes, mostrar_html, imagen_control,
    COLOR_BASE, COLOR_ACENTO, COLORES_GRUPO, MAPA_GRUPOS,
)

%matplotlib inline
%config InteractiveShell.ast_node_interactivity = "all"
configurar_estilo()  # Conserva el formato original, sin ajustar modelos.
SEED = 12345  # Semilla general; la comparación final también utiliza 1234.

### 5.0.1. Funciones para escalar, resumir y ajustar

- `escalar` resta la media y divide por la desviación típica **muestral** de cada columna (`ddof=1`). La entrada debe ser numérica, sin ausentes y sin columnas constantes.
- `resumen_particion` recibe los datos **en la escala del ajuste** y sus etiquetas: devuelve centroides, tamaños e inercias. No necesita clases conocidas.
- `ajustar_kmeans` permite elegir **Lloyd** o la implementación didáctica de transferencias **Hartigan–Wong**. No se garantiza identidad numérica con otra biblioteca o con R: inicialización, tolerancias y desempates también cuentan.

Los números de grupo se renumeran por orden de aparición; solo identifican conjuntos de observaciones. Las funciones se definen ahora y se utilizarán más adelante: todavía no se ha agrupado Iris.

In [ ]:
from s5_modelos import escalar, renumerar, resumen_particion, ajustar_kmeans
# Las funciones reciben sus datos explícitamente; importar no agrupa Iris.

### 5.0.2. Herramientas de representación

`representar_clusters` recibe **etiquetas ya calculadas** y proyecta las medidas en dos componentes principales (PCA). Por defecto estandariza **para dibujar**, no para volver a ajustar los grupos. `estandarizar=False` muestra PCA en la escala que se le entrega.

Los porcentajes de los ejes indican varianza conservada por cada componente; no son aciertos ni variabilidad explicada por los clústeres. Las envolventes convexas son contornos gráficos, no fronteras de decisión ni pruebas de separación en todas las dimensiones.

In [ ]:
from s5_graficos import representar_clusters, pares_coloreados
# Reciben etiquetas ya calculadas: la proyección no vuelve a ajustar los grupos.

## 5.1. Inspeccionar Iris y justificar la escala

1. Cargamos **150 flores**; cada fila contiene longitud y anchura de sépalo y pétalo, todas en **centímetros**.
2. Guardamos `Species` para evaluación externa y copiamos solo las cuatro medidas en `iris2`.
3. Revisamos dispersión y relaciones antes de calcular distancias. Una variable que varía más puede pesar más aunque todas midan centímetros.

Los ejemplos base ajustan con las **medidas originales**. `iris2_escalado` es una tabla aparte para explorar otra geometría. Estandarizar no es una mejora automática: da peso comparable a las desviaciones de cada variable y cambia la pregunta.

In [ ]:
dataset = load_iris()  # Carga las medidas y las etiquetas conocidas, en el orden original.
iris = pd.DataFrame(dataset.data, columns=["Sepal.Length", "Sepal.Width", "Petal.Length", "Petal.Width"])
iris.index = np.arange(1, len(iris) + 1)  # Identificadores de flores, no un orden temporal.
iris["Species"] = pd.Categorical.from_codes(dataset.target, dataset.target_names)  # Solo contraste externo.
iris  # Inspecciona las cuatro medidas (cm) y la columna de especie separada.

In [ ]:
iris2 = iris.iloc[:, :4].copy()  # Copia solo medidas: Species no entra en distancias ni ajustes.
iris2  # La copia conserva los identificadores de las mismas 150 flores.
display(iris2.describe().T[["count", "mean", "std", "min", "max"]].round(3))  # Comprueba escalas.
assert iris2.shape == (150, 4) and not iris2.isna().any().any()  # Evita un análisis con lectura incompleta.

### 5.1.1. Leer el resumen y los gráficos de pares

Las cuatro columnas tienen **150 valores válidos**. Sus desviaciones típicas son aproximadamente **0,828; 0,436; 1,765 y 0,762 cm**, en el orden de la tabla. `Petal.Length` varía mucho más que `Sepal.Width`: sus diferencias pueden dominar la distancia euclídea sin escalar.

En la matriz de pares, la columna del panel determina el eje horizontal y la fila el vertical. La diagonal muestra histogramas. Busca asociaciones y huecos entre nubes **sin colorear por especie**; una separación en dos medidas no demuestra que todas las flores formen tres grupos estables.

In [ ]:
# Cada punto es una flor; los histogramas resumen cada medida por separado.
_ = scatter_matrix(iris2, figsize=(9, 9), diagonal="hist", s=15, alpha=0.6)
plt.show()  # No se usan Species ni grupos calculados en esta inspección.

### 5.1.2. Mapas de calor: datos originales y estandarizados

El mapa reordena flores y variables según proximidad; **no** crea ni valida especies. Cambia únicamente la casilla de estandarización y compara los patrones de las columnas.

$$z_{ij}=\frac{x_{ij}-\bar{x}_j}{s_j}.$$

$x_{ij}$ es la medida de la flor $i$ en la variable $j$, en cm; $\bar{x}_j$ y $s_j$ son su media y desviación típica muestral. $z_{ij}$ no tiene unidades: $+1$ significa una desviación típica por encima de la media, no un centímetro adicional.

**Alternativa sin widget:** llama `mapa_iris(False)` y `mapa_iris(True)`. Estos mapas no sustituyen `iris2` ni el árbol base que construiremos después.

In [ ]:
from s5_graficos import mapa_iris as _mapa_iris

iris2_escalado = escalar(iris2)  # Tabla aparte; conserva las medidas originales.
mapa_iris = partial(_mapa_iris, iris2, iris2_escalado)  # Después solo se elige la escala de la vista.

In [ ]:
from s5_controles import crear_control_mapa, panel_interactivo

control_mapa = crear_control_mapa(mapa_iris)  # El control parte de las medidas originales.
display(panel_interactivo('Explora los mapas de calor de Iris', control_mapa))

**Lectura del mapa:** sin escalar, los colores comparan magnitudes en cm; tras estandarizar, comparan desviaciones respecto a la media de **cada** variable. El orden de las hojas puede cambiar porque también cambian las proximidades. Los árboles de columnas son una ayuda exploratoria: agrupan medidas, no flores adicionales.

#### Micropráctica 1 — comprobar la escala

Usa `std(ddof=1)`, ya empleado en `escalar`, para comparar la dispersión de `Petal.Length` en `iris2` y `iris2_escalado`. Completa el nombre de la columna. Antes de ejecutar, predice qué valor tendrá la desviación típica estandarizada y en qué unidades se expresa cada resultado.

In [ ]:
# Micropráctica opcional: no modifica las tablas ni se usa en los ejemplos siguientes.
variable_practica = ...  # Completa con la columna de longitud del pétalo.
if variable_practica is Ellipsis:
    print("Micropráctica pendiente: completa el nombre de la variable.")
else:
    display(pd.Series({"Desviación original (cm)": iris2[variable_practica].std(ddof=1),
                       "Desviación estandarizada (sin unidades)": iris2_escalado[variable_practica].std(ddof=1)}))
    # Explica por qué estandarizar no equivale a mejorar el acuerdo con Species.

## 5.2. Clustering jerárquico (diapositivas 8–18)

El algoritmo **aglomerativo** empieza con una flor por grupo y fusiona dos grupos en cada paso. Construir el árbol no exige fijar $k$; **cortarlo sí exige una decisión**. La alternativa divisiva empieza con un solo grupo y lo divide; aquí no se implementa.

Distinguimos dos decisiones:
- **Distancia entre observaciones:** euclídea (raíz de la suma de diferencias al cuadrado) o Manhattan (suma de diferencias absolutas).
- **Enlace entre grupos:** simple (mínimo de pares), completo (máximo), promedio (media de todos los pares entre ambos grupos) o Ward (mínimo aumento de inercia euclídea).

### 5.2.1. Matriz de distancias

`pdist` recibe **solo** `iris2` y calcula $150\cdot149/2=11\,175$ distancias sin duplicar pares. `squareform` las convierte en una matriz simétrica de 150 × 150, con ceros en la diagonal.

La base usa **distancia euclídea sobre las cuatro medidas originales**, en cm. Cambiar la escala relativa de las variables o la métrica cambia las proximidades. La matriz ocupa memoria cuadrática: no trasladar este procedimiento sin valorar el coste a millones de filas.

In [ ]:
d = pdist(iris2, metric="euclidean")  # Vector de distancias entre pares, en cm.
matriz_distancias = pd.DataFrame(squareform(d), index=iris2.index, columns=iris2.index)
matriz_distancias  # Filas y columnas identifican las mismas flores.
print(f"Pares: {len(d):,}; distancia mínima: {d.min():.3f} cm; máxima: {d.max():.3f} cm")

**Lectura de las distancias:** hay **11 175 pares** y el máximo es **7,085 cm**. El mínimo entre dos filas es **0**: algunas flores tienen las mismas cuatro medidas registradas. Un cero fuera de la diagonal no es necesariamente un error ni justifica eliminar datos.

### 5.2.2. Jerarquización con enlace completo

`linkage(d, method="complete")` construye el árbol usando el **máximo** de las distancias entre miembros de dos grupos. `hc` tiene 149 fusiones; sus columnas identifican las dos ramas unidas, su altura y cuántas flores reúne la nueva rama. No contiene especies ni una elección de $k$.

In [ ]:
hc = linkage(d, method="complete")  # Construye 149 fusiones a partir de las distancias originales.
# Conserva hc: los controles crearán otros árboles sin sustituir este referente.

### 5.2.3. Leer el dendrograma

Cada fusión une ramas a una **altura de disimilitud**. Con enlace completo, la altura es la mayor distancia entre los miembros de los dos grupos unidos, en cm; no es su tamaño. El orden horizontal de las hojas se puede permutar sin cambiar la jerarquía.

**Ward no usa ese mismo criterio.** Para grupos $A$ y $B$:

$$\Delta W=\frac{n_A n_B}{n_A+n_B}\lVert\bar{x}_A-\bar{x}_B\rVert^2.$$

$n_A,n_B$ son tamaños y $\bar{x}_A,\bar{x}_B$ sus centroides. $\Delta W$ es el aumento de la inercia intra (cm² en Iris original). SciPy dibuja la altura Ward como $\sqrt{2\Delta W}$, no como $\Delta W$ directamente. **Ward requiere disimilitudes euclídeas.**

**Para pensar:** un hueco vertical puede sugerir un corte, pero ¿se mantendría al cambiar el peso relativo de las variables?

In [ ]:
from s5_graficos import dibujar_dendrograma

dibujar_dendrograma(hc, iris2.index)  # Árbol fijo y etiquetas de las mismas 150 flores.

**Lectura del árbol fijo:** la última fusión llega a **7,085 cm**, coherente con el máximo de `d`. Las fusiones bajas indican proximidad en las medidas utilizadas; ni el color ni la posición horizontal identifican especies.

#### Explorar escala, distancia y enlace

Cambia **una decisión a la vez**. El control calcula un árbol nuevo y su correlación cofenética frente a **sus propias** distancias. Una correlación alta describe fidelidad de la jerarquía, no un número verdadero de grupos.

La correlación es adimensional: cambiar todas las unidades por un mismo factor no la altera. Cambiar **pesos relativos**, variables o métrica sí cambia la geometría de referencia; no elijas un modelo solo porque su correlación sea mayor.

**Restricción:** se impide combinar Ward con Manhattan. Este experimento no modifica `hc` ni `ct`. **Alternativa estática:** `display(widgets.HTML(value=explorar_arbol("euclidean", "complete", False)))`. Para una segunda vista válida, cambia solo el enlace a `"ward"`.

In [ ]:
def explorar_arbol(distancia="euclidean", enlace="complete", estandarizar=False):
    """Explora un árbol nuevo sin modificar hc, ct ni los resultados fijos."""
    # 1. Verifica la compatibilidad de Ward y prepara la geometría solicitada.
    if enlace == "ward" and distancia != "euclidean":
        raise ValueError("Para interpretar Ward como incremento de inercia usa distancia euclídea.")
    tabla = escalar(iris2) if estandarizar else iris2
    disimilitudes = pdist(tabla, metric=distancia)
    arbol = linkage(disimilitudes, method=enlace)
    calidad = cophenet(arbol, disimilitudes)[0]
    # 2. Dibuja el árbol nuevo y compara su representación con sus propias distancias.
    fig, ax = plt.subplots(figsize=(12, 4))
    _ = dendrogram(arbol, no_labels=True, ax=ax, color_threshold=0,
                   above_threshold_color=COLOR_BASE)
    _ = ax.set(title=f"{enlace.title()} · {distancia} · "
                     f"{'estandarizado' if estandarizar else 'sin escalar'}",
               xlabel="Flores ordenadas por el árbol", ylabel="Altura de la fusión")
    fig.tight_layout()
    return imagen_control(fig, "Dendrograma comparativo de Iris") + (
        f"<p>Correlación cofenética con su propia distancia: <b>{calidad:.3f}</b>. "
        "Un valor alto no demuestra que exista un número verdadero de grupos.</p>")


# 3. Controles originales; la escala solo afecta a este experimento.
selector_distancia = widgets.Dropdown(options=[("Euclídea", "euclidean"),
                                               ("Manhattan", "cityblock")],
                                     value="euclidean", description="Distancia")
selector_enlace = widgets.Dropdown(options=[("Completo", "complete"),
                                            ("Promedio", "average"),
                                            ("Simple", "single"), ("Ward", "ward")],
                                  value="complete", description="Enlace")
selector_escala = widgets.Checkbox(value=False, description="Estandarizar")
visor_arbol = widgets.HTML(layout=widgets.Layout(width="100%"))


def actualizar_arbol(cambio=None):
    # Evita Ward no euclídeo y deja que el observador actualice la figura una sola vez.
    if cambio is not None and cambio["name"] != "value":
        return
    if selector_distancia.value != "euclidean" and selector_enlace.value == "ward":
        selector_enlace.value = "complete"
        return  # El cambio de enlace llama al observador otra vez.
    visor_arbol.value = explorar_arbol(selector_distancia.value, selector_enlace.value,
                                      selector_escala.value)


for control in (selector_distancia, selector_enlace, selector_escala):
    control.observe(actualizar_arbol, names="value")
actualizar_arbol()  # Muestra la combinación inicial sin esperar a mover un control.
control_arbol = widgets.VBox([
    cabecera_widget("Explora distancias y enlaces"),
    fila_controles([selector_distancia, selector_enlace, selector_escala]),
    visor_arbol,
], layout=widgets.Layout(width="100%", grid_gap="12px"))
display(control_arbol)

### 5.2.4. Calidad del árbol: correlación cofenética (diapositiva 17)

La **distancia cofenética** entre dos flores es la altura de su primera rama común. Comparamos las 11 175 alturas con las distancias originales del mismo árbol.

`np.corrcoef` calcula una **correlación de Pearson**: resume asociación lineal, no igualdad exacta entre distancias ni solamente orden de proximidades. Un valor alto no prueba grupos naturales ni calidad del corte en tres.

La referencia **0,75** de la presentación es orientativa, no un umbral universal ni un nivel de significación.

In [ ]:
cop_dist = cophenet(hc)  # Altura de la primera unión de cada par, en el mismo orden que d.
np.corrcoef(d, cop_dist)[0, 1]  # Pearson entre distancia original y distancia cofenética.

**Lectura de la correlación:** el enlace completo da **0,727**, algo por debajo de la referencia 0,75. La jerarquía reproduce parte de la estructura de distancias, pero no de forma perfecta. Esto no obliga a descartarla ni valida ningún corte.

### 5.2.5. Explorar particiones: inercia y codo (diapositivas 12–13 y 18)

Cortar el árbol en $k$ grupos permite calcular la **inercia intra**:

$$W_k=\sum_{g=1}^{k}\sum_{i\in g}\lVert x_i-\bar{x}_g\rVert^2,\qquad T=W_k+B_k.$$

$x_i$ contiene las cuatro medidas de una flor y $\bar{x}_g$ es el centro de su grupo. $T$ suma distancias cuadráticas a la media global; $B_k$ es la inercia entre grupos. En Iris original, todas las inercias se expresan en **cm²**.

La proporción $B_k/T=1-W_k/T$ mide reducción de dispersión respecto a un único grupo: es adimensional y **no** mide clasificación correcta. Aumenta al refinar cortes, incluso sin grupos útiles.

**Regla del codo:** busca una caída fuerte seguida de mejoras menores. La inercia se calcula con centroides euclídeos aunque el árbol use enlace completo; su algoritmo no está minimizando directamente esta curva. Compara también tamaño, estabilidad e interpretación, no solo el codo.

In [ ]:
# Corta el MISMO árbol para k=1,...,10 y resume cada partición en la escala original.
inercia_hc = pd.Series({k: resumen_particion(iris2, cut_tree(hc, n_clusters=k).ravel()).inertia_
                        for k in range(1, 11)}, name="Inercia intra")
ax = inercia_hc.plot(marker="o", color=COLOR_BASE)
_ = ax.set(xlabel="Número de grupos", ylabel="Inercia intra (cm²)", title="Regla del codo")
_ = estilo_ejes(ax, cuadricula=True)
plt.show()
display(inercia_hc.loc[1:4].round(3))  # Hace explícitas las primeras caídas del gráfico.

**Lectura del codo jerárquico:** $W_1=681,371$, $W_2=235,153$, $W_3=89,525$ y $W_4=60,973$ cm². Las primeras reducciones son fuertes y después se atenúan: tres grupos es un referente didáctico razonable, **no** una verdad demostrada.

### 5.2.6. Comparar 26 indicadores internos

La rutina conserva 26 familias inspiradas en NbClust: KL, CH, Hartigan, CCC, Scott, Marriott, TraceCovW, TraceW, Friedman, Rubin, C-index, DB, silhouette, Duda, pseudo-t², Beale, Ratkowsky, Ball, punto-biserial, Frey, McClain, Dunn, Hubert, SD, D-index y SDbw. **Hubert y D-index son diagnósticos gráficos y no votan**; las recomendaciones ausentes tampoco.

La **silhouette** compara la distancia media de la flor $i$ a su grupo, $a(i)$, con la menor distancia media a otro grupo, $b(i)$:

$$s(i)=\frac{b(i)-a(i)}{\max\{a(i),b(i)\}}.$$

Es adimensional: cerca de **1**, buena separación; cerca de **0**, frontera; negativa, mayor proximidad media a otro grupo. En un grupo de una sola observación, scikit-learn asigna 0. **Dunn** divide la separación mínima entre grupos por el mayor diámetro interno: cuanto mayor, mejor según ese criterio, pero un solo punto puede determinarlo.

**Alcance de la rutina:** es didáctica, no una réplica certificada de NbClust. DB usa dispersión **RMS**, distinta de la distancia media empleada por scikit-learn; algunos índices históricos usan productos de datos **sin centrar**, no el `totss` escalar anterior. Duda/pseudo-t²/Beale examinan una división local del grupo más disperso. Las reglas de diferencias se aplican como muestra el código; sus votos no son pruebas independientes ni p-valores para $k$.

En la siguiente llamada evaluamos **Ward euclídeo**, no el árbol completo `hc`. No atribuyas sus recomendaciones al otro enlace. Ninguna mayoría garantiza el número correcto de grupos.

In [ ]:
def indice_dunn(distancias, etiquetas):
    # Razón adimensional: menor separación entre grupos / mayor diámetro dentro de ellos.
    D = np.asarray(distancias)
    grupos = np.unique(etiquetas)
    diametro = max(D[np.ix_(etiquetas == g, etiquetas == g)].max() for g in grupos)
    separacion = min(D[np.ix_(etiquetas == g, etiquetas == h)].min()
                     for i, g in enumerate(grupos) for h in grupos[i+1:])
    return separacion / diametro if diametro > 0 else np.inf


def evaluar_indices(datos, metodo="ward", min_k=2, max_k=15, random_state=12345):
    """26 criterios didácticos: 24 reglas de elección y dos diagnósticos gráficos."""
    # 1. Prepara una sola matriz de distancias y todas las particiones necesarias.
    a = np.asarray(datos, dtype=float)
    n, p = a.shape
    D = squareform(pdist(a))
    superiores = np.triu_indices(n, 1)
    distancias = D[superiores]
    ordenadas = np.sort(distancias)
    particiones = {1: np.zeros(n, dtype=int)}
    arbol = None if metodo == "kmeans" else linkage(pdist(a), method=metodo)
    for k in range(2, max_k + 2):
        particiones[k] = (ajustar_kmeans(a, k, n_init=1, random_state=random_state).labels_
                          if metodo == "kmeans" else renumerar(cut_tree(arbol, n_clusters=k).ravel()))
    # T es aquí una MATRIZ histórica sin centrar; no confundir con el totss escalar.
    total_centrado = np.sum((a - a.mean(axis=0)) ** 2)
    T = a.T @ a
    s = np.sqrt(np.maximum(np.linalg.eigvalsh(T / (n - 1))[::-1], 1e-15))
    determinante_T = np.linalg.slogdet(T)[1]
    cache = {}
    # 2. Reutiliza centroides, residuos, pares internos/externos y densidades por partición.
    for k, etiquetas in particiones.items():
        r = resumen_particion(a, etiquetas)
        etiquetas = r.labels_
        residuos = a - r.cluster_centers_[etiquetas]
        W = residuos.T @ residuos
        B = T - W
        mismas = etiquetas[superiores[0]] == etiquetas[superiores[1]]
        dentro, fuera = distancias[mismas], distancias[~mismas]
        var_grupos = np.array([a[etiquetas == j].var(axis=0) for j in range(k)])
        norma_var = np.linalg.norm(var_grupos, axis=1)
        scatt = norma_var.mean() / np.linalg.norm(a.var(axis=0))
        radio = np.sqrt(norma_var.sum()) / k
        centros_D = squareform(pdist(r.cluster_centers_))
        dis = ((pdist(r.cluster_centers_).max() / pdist(r.cluster_centers_).min())
               * np.sum(1 / centros_D.sum(axis=1))) if k > 1 else np.nan
        cache[k] = dict(r=r, labels=etiquetas, W=W, B=B, dentro=dentro, fuera=fuera,
                        scatt=scatt, radio=radio, dis=dis, centros_D=centros_D)

    filas, criticos = [], []
    for k in range(min_k, max_k + 1):
        c = cache[k]
        r, etiquetas, W, B = c["r"], c["labels"], c["W"], c["B"]
        previo, siguiente = cache[k-1]["r"], cache[k+1]["r"]
        dentro, fuera = c["dentro"], c["fuera"]
        intra = r.inertia_
        signo, logdetW = np.linalg.slogdet(W)
        # 3. DB conserva la versión RMS; C-index usa las sumas de pares extremos.
        dispersion = np.sqrt(r.withinss / r.size)
        ratios = (dispersion[:, None] + dispersion[None, :]) / np.where(c["centros_D"] > 0, c["centros_D"], np.inf)
        m = len(dentro)
        minimo, maximo = ordenadas[:m].sum(), ordenadas[-m:].sum()
        # CCC conserva la referencia histórica de T sin centrar: no es B_k/T del codo.
        u = s / (np.prod(s) / k) ** (1 / p)
        p1 = min(int((u >= 1).sum()), k - 1)
        if 0 < p1 < p:
            u = s / (np.prod(s[:p1]) / k) ** (1 / p1)
            suma = np.sum(1 / (n + u[:p1])) + np.sum(u[p1:] ** 2 / (n + u[p1:]))
            dimension = p1
        else:
            suma, dimension = np.sum(1 / (n + u)), p
        esperado_R2 = 1 - suma / np.sum(u ** 2) * (n - k) ** 2 / n * (1 + 4 / n)
        R2 = 1 - np.trace(W) / np.trace(T)
        ccc = (np.log((1 - esperado_R2) / (1 - R2)) * np.sqrt(n * dimension / 2)
               / (0.001 + esperado_R2) ** 1.2)

        # 4. División LOCAL en dos del grupo más disperso, no contraste global sobre k.
        padre = int(r.withinss.argmax())
        local = a[etiquetas == padre]
        division = ajustar_kmeans(local, 2, n_init=1, random_state=random_state)
        wl = division.inertia_
        wm = r.withinss[padre]
        nm = len(local)
        duda = wl / wm
        # Corrección mínima: el término 8/(pi²*p) también incluye la dimensión p.
        critica_duda = 1 - 2 / (np.pi * p) - 3.2 * np.sqrt(2 * (1 - 8 / (np.pi ** 2 * p)) / (nm * p))
        pseudo = (wm - wl) / (wl / (nm - 2))
        beale = ((wm - wl) / wl) / ((nm - 1) / (nm - 2) * (2 ** (2 / p) - 1))
        criticos.append({"k": k, "Duda": critica_duda,
                         "PseudoT2": (1 - critica_duda) / critica_duda * (nm - 2),
                         "Beale": stats.f.sf(beale, p, (nm - 2) * p)})

        # 5. Densidades alrededor de centros y puntos medios; Hubert es solo diagnóstico.
        dist_centro = np.linalg.norm(a - r.cluster_centers_[etiquetas], axis=1)
        densidades = np.bincount(etiquetas, weights=(dist_centro <= c["radio"]), minlength=k)
        dens_bw = 0.0
        for i in range(k):
            for j in range(i+1, k):
                candidatos = a[(etiquetas == i) | (etiquetas == j)]
                centro = (r.cluster_centers_[i] + r.cluster_centers_[j]) / 2
                cantidad = (np.linalg.norm(candidatos - centro, axis=1) <= c["radio"]).sum()
                denominador = max(densidades[i], densidades[j])
                dens_bw += cantidad / denominador if denominador else 0
        Q = cdist(r.cluster_centers_[etiquetas], r.cluster_centers_[etiquetas])
        varP = np.linalg.norm(D.var(axis=0))
        varQ = np.linalg.norm(Q.var(axis=0))
        hubert = np.sum((D - D.mean())[superiores] * (Q - Q.mean())[superiores]) / (len(distancias) * varP * varQ)
        proxima = cache[k+1]
        denominador_frey = proxima["dentro"].mean() - dentro.mean()
        kl_num = (k-1) ** (2/p) * previo.inertia_ - k ** (2/p) * intra
        kl_den = k ** (2/p) * intra - (k+1) ** (2/p) * siguiente.inertia_
        # 6. Reúne criterios. solve(W, B) calcula W⁻¹B, no multiplicación elemento a elemento.
        filas.append({
            "k": k, "KL": abs(kl_num / kl_den),
            "CH": ((total_centrado - intra) / (k-1)) / (intra / (n-k)),
            "Hartigan": (intra / siguiente.inertia_ - 1) * (n-k), "CCC": ccc,
            "Scott": n * (determinante_T - logdetW), "Marriott": k ** 2 * np.linalg.det(W),
            "TraceCovW": np.var(W, axis=0, ddof=1).sum(), "TraceW": np.trace(W),
            "Friedman": np.trace(np.linalg.solve(W, B)), "Rubin": np.trace(T) / np.trace(W),
            "Cindex": (dentro.sum() - minimo) / (maximo - minimo), "DB": ratios.max(axis=1).mean(),
            "Silhouette": silhouette_score(D, etiquetas, metric="precomputed"),
            "Duda": duda, "PseudoT2": pseudo, "Beale": beale,
            "Ratkowsky": np.sqrt(1 - np.sum((a - r.cluster_centers_[etiquetas]) ** 2, axis=0)
                                    / np.sum((a - a.mean(axis=0)) ** 2, axis=0)).mean() / np.sqrt(k),
            "Ball": intra / k,
            "PointBiserial": (fuera.mean() - dentro.mean()) * np.sqrt(len(fuera)*len(dentro))
                              / len(distancias) / distancias.std(ddof=1),
            "Frey": (proxima["fuera"].mean() - fuera.mean()) / denominador_frey,
            "McClain": (dentro.mean()/len(dentro)) / (fuera.mean()/len(fuera)),
            "Dunn": indice_dunn(D, etiquetas), "Hubert": hubert,
            "SDindex": cache[max_k]["dis"] * c["scatt"] + c["dis"],
            "Dindex": dist_centro.mean(), "SDbw": c["scatt"] + 2 * dens_bw / (k * (k-1)),
        })
    indices = pd.DataFrame(filas).set_index("k")
    criticos = pd.DataFrame(criticos).set_index("k")
    # 7. Aplica máximos, mínimos, diferencias y reglas locales; no todos se minimizan.
    recomendaciones = {}
    maximos = ["KL", "CH", "CCC", "Silhouette", "Ratkowsky", "PointBiserial", "Dunn"]
    minimos = ["Cindex", "DB", "McClain", "SDindex", "SDbw"]
    for nombre in maximos:
        recomendaciones[nombre] = indices[nombre].idxmax()
    for nombre in minimos:
        recomendaciones[nombre] = indices[nombre].idxmin()
    for nombre in ["Hartigan", "Scott", "TraceCovW", "Friedman", "Ball"]:
        recomendaciones[nombre] = (indices[nombre] - indices[nombre].shift(-1)).abs().idxmax()
    for nombre, maximizar in [("Marriott", True), ("TraceW", True), ("Rubin", False)]:
        diferencias = indices[nombre].shift(1) - 2 * indices[nombre] + indices[nombre].shift(-1)
        recomendaciones[nombre] = diferencias.idxmax() if maximizar else diferencias.idxmin()
    condiciones = {"Duda": indices["Duda"] >= criticos["Duda"],
                   "PseudoT2": indices["PseudoT2"] <= criticos["PseudoT2"],
                   "Beale": criticos["Beale"] >= 0.1, "Frey": indices["Frey"] < 1}
    for nombre, condicion in condiciones.items():
        candidatos = indices.index[condicion]
        recomendaciones[nombre] = (max(min_k, int(candidatos[0]) - 1) if nombre == "Frey" else candidatos[0]) if len(candidatos) else np.nan
    # 8. Cuenta solo recomendaciones válidas y devuelve una partición orientativa, no verdadera.
    best = pd.Series(recomendaciones, name="k recomendado").astype("Int64")
    votos = best.dropna().value_counts().sort_index()
    ganador = int(votos.idxmax())
    return SimpleNamespace(All_index=indices, Best_n=best.to_frame(), Votes=votos,
                           Best_partition=particiones[ganador], Critical=criticos, partitions=particiones)

In [ ]:
# Evalúa cortes de un árbol WARD euclídeo, con las medidas originales y k=2,...,15.
NB = evaluar_indices(iris2, metodo="ward", min_k=2, max_k=15, random_state=SEED)
NB.Best_n  # Una recomendación por criterio válido; no utiliza Species.
display(NB.Votes.rename("Número de recomendaciones"))

In [ ]:
ax = NB.Votes.plot.bar(rot=0, color=COLOR_BASE)
_ = ax.set(xlabel="Número de grupos", ylabel="Votos", title="Recomendaciones de los indicadores")
_ = estilo_ejes(ax, cuadricula=True)
plt.show()

**Lectura de los indicadores ejecutados:** en Ward, 12 de las 24 recomendaciones válidas favorecen $k=2$, 6 favorecen $k=3$ y las otras se reparten entre 6, 11 y 15. No hay unanimidad; los criterios comparten información y no son 24 contrastes independientes. La mayoría por dos grupos no invalida estudiar un corte en tres para describir una resolución distinta.

### 5.2.7. Cortar el árbol completo en tres grupos

1. Volvemos a **`hc`, el enlace completo sin escalar**, no a la partición Ward de `NB`.
2. `cut_tree(..., n_clusters=3).ravel()` devuelve una etiqueta por flor; `renumerar` solo cambia sus nombres.
3. Guardamos `ct` como referente fijo. El control calcula otros cortes en variables locales y los rectángulos marcan ramas, no especies.
4. Resumimos tamaños e inercia con las mismas cuatro medidas originales.

**Lectura del corte fijo:** los tamaños son **50, 72 y 28** y $B_3/T=86,9\%$. Ese porcentaje es dispersión reducida respecto a un solo centro, **no flores correctamente clasificadas**. Aumentar $k$ suele elevarlo sin demostrar grupos naturales.

**Alternativa estática:** `display(particion_jerarquica(3))`; el valor devuelto es la serie de tamaños. El deslizador modifica la vista, no `ct`.

In [ ]:
# Referente fijo: enlace completo, distancia euclídea y cuatro medidas originales.
ct = pd.Series(renumerar(cut_tree(hc, n_clusters=3).ravel()), index=iris.index, name="grupo")


def particion_jerarquica(k):
    # 1. Corta el mismo árbol sin sustituir ct; coloca las hojas según el dendrograma.
    grupos = renumerar(cut_tree(hc, n_clusters=k).ravel())
    fig, ax = plt.subplots(figsize=(14, 5))
    dibujado = dendrogram(hc, no_labels=True, link_color_func=lambda nodo: "#5B6770", ax=ax)
    orden = np.asarray(dibujado["leaves"])
    altura = (hc[-k, 2] + hc[-k+1, 2]) / 2  # Entre las dos fusiones que delimitan este corte.
    # 2. Delimita las hojas de cada grupo; el ancho representa cuántas hojas reúne.
    for grupo in np.unique(grupos):
        posiciones = np.flatnonzero(grupos[orden] == grupo)
        color = COLORES_GRUPO[int(grupo) % 10]
        ax.add_patch(Rectangle((10 * posiciones.min(), 0), 10 * len(posiciones), altura,
                               fill=False, edgecolor=color, linewidth=2))
    ax.set(xlabel="Observaciones ordenadas por el árbol", ylabel="Distancia (cm)", title=f"Partición en {k} grupos")
    fig.tight_layout()
    display(widgets.HTML(value=imagen_control(fig, f"Dendrograma dividido en {k} grupos")))
    # 3. Evalúa este corte en la escala ORIGINAL, no sobre la imagen ni contra Species.
    resumen = resumen_particion(iris2, grupos)
    print(f"Reducción relativa de dispersión B/T (no acierto): {resumen.betweenss / resumen.totss:.1%}")
    return pd.Series(np.bincount(grupos), index=np.arange(1, k+1), name="Tamaño del grupo")

In [ ]:
control_jerarquia = widgets.interactive(
    particion_jerarquica,
    k=widgets.IntSlider(value=3, min=2, max=8, description="Grupos", continuous_update=False),
)
display(panel_interactivo("Explora el corte del árbol", control_jerarquia))

### 5.2.8. ACP + clustering y consolidación: HCPC (diapositivas 38–39)

1. Estandarizamos las cuatro medidas y calculamos **todas** las componentes principales (ACP/PCA).
2. En sus coordenadas construimos un árbol de **enlace completo**, tal como conserva este ejemplo. No es el Ward de `NB` ni una réplica exacta de HCPC de otra biblioteca.
3. Cortamos en $k$ grupos. Si consolidamos, iniciamos Hartigan–Wong **en los centroides de ese corte** y permitimos transferencias que reduzcan la WSS.
4. Dibujamos solo dos componentes, aunque ajustamos con las cuatro.

**¿Reduce PCA la dimensión aquí? No.** Con todas las componentes, la rotación conserva las distancias euclídeas de los datos **estandarizados**. No conserva la geometría de Iris sin escalar. Retener menos componentes cambiaría las distancias y podría retirar tanto ruido como señal.

**API:** `hcpc(k, consolidar)` devuelve `labels_`, `tree_` y `scores_`; `explorar_hcpc` actualiza `HC` para el widget. Ninguno sustituye `ct`. Los centros explícitos suponen un único inicio: consolidar no busca entre diez arranques nuevos ni asegura el mínimo global.

**Alternativa estática y comprobación:** las llamadas fijas siguientes comparan ambas opciones en la misma escala. La WSS consolidada no debe aumentar; eso no significa mayor coincidencia con `Species`.

In [ ]:
def hcpc(k, consolidar):
    # 1. ACP completa sobre Iris ESTANDARIZADO: rota, pero no reduce la dimensión.
    coordenadas = PCA().fit_transform(iris2_escalado)
    arbol = linkage(pdist(coordenadas), method="complete")
    inicial = renumerar(cut_tree(arbol, n_clusters=k).ravel())
    # 2. Los centroides del corte son el inicio explícito de la consolidación.
    centros = np.array([coordenadas[inicial == j].mean(axis=0) for j in range(k)])
    grupos = ajustar_kmeans(coordenadas, k, init=centros).labels_ if consolidar else inicial
    # 3. La figura usa dos componentes; las etiquetas se calcularon con TODAS.
    grafico = representar_clusters(iris2_escalado, grupos, titulo="Clustering sobre componentes principales",
                                   estandarizar=False, mostrar=False)
    display(widgets.HTML(value=grafico))
    return SimpleNamespace(labels_=grupos, tree_=arbol, scores_=coordenadas)


def explorar_hcpc(k, consolidar):
    # El control mantiene su resultado en HC; ct y los ajustes fijos siguen intactos.
    global HC
    HC = hcpc(k, consolidar)

In [ ]:
# El widget parte de tres grupos consolidados y conserva las dos decisiones explorables.
control_hcpc = widgets.interactive(
    explorar_hcpc,
    k=widgets.IntSlider(value=3, min=2, max=8, description="Grupos", continuous_update=False),
    consolidar=widgets.Checkbox(value=True, description="Consolidar con k-means"),
)
display(panel_interactivo("Explora el árbol sobre componentes principales", control_hcpc))

In [ ]:
# Ejemplo fijo, independiente de la última posición del widget: misma escala y mismo k.
hcpc_inicial = hcpc(3, False)
hcpc_consolidado = hcpc(3, True)
resumen_hcpc = pd.DataFrame({
    nombre: {"WSS (escala estandarizada)": resumen_particion(resultado.scores_, resultado.labels_).inertia_,
             "B/T (no acierto)": resumen_particion(resultado.scores_, resultado.labels_).betweenss /
                                resumen_particion(resultado.scores_, resultado.labels_).totss,
             "Tamaños": tuple(np.bincount(resultado.labels_))}
    for nombre, resultado in [("Corte inicial", hcpc_inicial), ("Consolidado", hcpc_consolidado)]
}).T
display(resumen_hcpc)
assert resumen_hcpc.loc["Consolidado", "WSS (escala estandarizada)"] <= resumen_hcpc.loc[
    "Corte inicial", "WSS (escala estandarizada)"] + 1e-9
assert np.allclose(pdist(hcpc_inicial.scores_), pdist(iris2_escalado))  # ACP completa conserva distancias.
print("La consolidación reduce o mantiene WSS; ni la rotación ni B/T certifican especies.")

**Lectura de HCPC:** la tabla compara el árbol inicial y la consolidación en coordenadas estandarizadas, no con la WSS en cm² del corte `ct`. Los tamaños permiten localizar redistribuciones. Que WSS baje solo indica mejor ajuste del criterio de centroides desde ese inicio; no demuestra clases verdaderas ni mejora externa.

### 5.2.9. Representar los tres grupos jerárquicos fijos

1. Recuperamos **`ct`**, obtenido con medidas originales y enlace completo.
2. En los gráficos de pares conservamos las cuatro medidas en cm y coloreamos por grupo descubierto.
3. Después comparamos dos proyecciones ACP: sin escalar y estandarizada **solo para dibujar**.

**Qué mirar:** busca en qué pares se separan o solapan colores. Una misma flor conserva su grupo en todos los paneles; cambiar el plano o escalar la proyección no recalcula la partición. Los colores son nombres arbitrarios, no especies.

In [ ]:
# Grupos del enlace completo SIN escalar; ejes de cada pareja en centímetros.
pares_coloreados(iris2, ct.to_numpy())  # La función dibuja las etiquetas recibidas, no las ajusta.

#### ACP de las medidas originales: mostrar dos ejes, no cambiar grupos

`PCA().fit(iris2)` centra las medidas, pero **no** las estandariza. CP1 conserva **92,46 %** de la varianza original y CP2 **5,31 %**; juntas, aproximadamente **97,77 %**. Son porcentajes de la **proyección**, no el 86,9 % de reducción de dispersión de los clústeres ni porcentajes correctos.

Las medidas de pétalo, con mayor variación, pesan mucho en esta ACP. Las dos coordenadas de cada punto son combinaciones de las cuatro medidas; no son sépalo y pétalo directamente.

In [ ]:
# 1. Ajusta ACP en las medidas originales y obtiene coordenadas de las mismas flores.
pr = PCA().fit(iris2)  # PCA centra; no aplica escalar automáticamente.
scores = pr.transform(iris2)
x, y = scores[:, 0], scores[:, 1]
# 2. Colorea con ct, sin volver a hacer clustering ni consultar Species.
fig, ax = plt.subplots()
_ = ax.scatter(x, y, c=ct, cmap=MAPA_GRUPOS, vmin=-0.5, vmax=9.5, s=24)
_ = ax.set(xlabel="CP1", ylabel="CP2", title="Clustering jerárquico en componentes principales")
fig.tight_layout()
plt.show()
display(pd.Series(100 * pr.explained_variance_ratio_[:2], index=["CP1", "CP2"],
                  name="Varianza de la proyección (%)").round(2))

#### ACP estandarizada y envolventes: otra vista de la misma partición

`representar_clusters` estandariza **para representar** antes de calcular dos componentes. Estas retienen **72,96 %** y **22,85 %** de la varianza estandarizada (aproximadamente **95,81 %** juntas). Siguen coloreando exactamente `ct`, calculado sin escalar.

**Lectura conjunta:** las envolventes pueden solaparse en el plano y ocultan dimensiones no representadas; no son fronteras aprendidas. Para cambiar el ajuste por estandarización habría que construir nuevas distancias y nuevos grupos, como en HCPC, no solo llamar a esta representación.

In [ ]:
# Estandarización solo VISUAL por defecto: conserva el corte ct de Iris original.
representar_clusters(iris2, ct.to_numpy(), titulo="Tres grupos jerárquicos")

### 5.2.10. Correspondencia posterior con especies (diapositivas 5 y 22)

**Ahora, y solo ahora, evaluamos externamente.** `Species` no intervino en distancias, enlace ni corte. No se puede usar esta tabla para escoger retrospectivamente $k$ o parámetros.

1. Cruzamos especie conocida y grupo descubierto con `pd.crosstab`.
2. Como los identificadores son arbitrarios, `linear_sum_assignment` busca una correspondencia **uno a uno** que maximice la diagonal; el signo negativo transforma esa maximización en minimización.
3. `cl` renombra los grupos para describir la coincidencia y `t_pred` muestra la tabla reordenada. No se ha entrenado un clasificador ni hecho predicciones sobre flores nuevas.
4. Comparamos también ARI, que trabaja con parejas y no requiere renombrar.

**Lectura ejecutada:** los 50 casos de setosa quedan juntos en este corte; los otros dos grupos mezclan especies. La coincidencia descriptiva tras la correspondencia es **126/150 = 84 %**, mientras que **ARI = 0,642**. No son el mismo indicador: 84 % no es exactitud de test ni el 86,9 % de dispersión reducida.

En la última matriz, el **símbolo** representa la especie y el **color** el grupo renombrado. Que distintos símbolos compartan color hace visible el desacuerdo. Esta asignación uno a uno es específica de tres grupos y tres especies; no la generalices sin revisar el criterio a otro $k$.

In [ ]:
# Inspección EXTERNA de una partición ya fijada: no modifica ct ni selecciona k.
tabla_especies = pd.crosstab(iris["Species"], ct + 1)
tabla_especies  # Cada columna cuenta flores del mismo grupo; Species solo describe las filas.

In [ ]:
# 1. Maximiza coincidencias mediante una correspondencia uno a uno, solo después del ajuste.
filas, columnas = linear_sum_assignment(-tabla_especies.to_numpy())
correspondencia = {int(columna): int(fila) for fila, columna in zip(filas, columnas)}
cl = ct.map(correspondencia)  # Renombra, pero no mueve ninguna flor de su grupo.
# 2. Cuenta la coincidencia descriptiva de ESTA muestra; no es validación en test.
t_pred = pd.crosstab(iris["Species"], cl + 1)
t_pred
np.trace(t_pred.to_numpy()) / t_pred.to_numpy().sum()
print(f"ARI jerárquico / especies (sin renombrar): {adjusted_rand_score(ct, dataset.target):.3f}")

In [ ]:
# Evaluación visual posterior: colores = grupos renombrados; símbolos = especies conocidas.
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
marcadores = ["o", "s", "^"]
for i, nombre_y in enumerate(iris2.columns):
    for j, nombre_x in enumerate(iris2.columns):
        ax = axes[i, j]
        if i == j:
            _ = ax.hist(iris2[nombre_x], bins=12, color=COLOR_BASE, alpha=0.75)
        else:
            # Species selecciona símbolos SOLO para dibujar: no reajusta ni elige grupos.
            for especie, marcador in enumerate(marcadores):
                mascara = dataset.target == especie
                _ = ax.scatter(iris2.loc[mascara, nombre_x], iris2.loc[mascara, nombre_y],
                               c=cl.loc[mascara], cmap=MAPA_GRUPOS, vmin=-0.5, vmax=9.5,
                               marker=marcador, s=12)
        if i == 3: _ = ax.set_xlabel(nombre_x)
        if j == 0: _ = ax.set_ylabel(nombre_y)
fig.tight_layout()
plt.show()  # Compara mezcla de símbolos dentro de cada color, no solo separación visual.

### 5.2.11. Comparar dendrogramas sin confundir acuerdo y fidelidad

1. Construimos árboles completo y promedio con **el mismo vector `d`**.
2. El `tanglegram` alinea hojas y une la **misma flor** entre ambos árboles. Los cruces dependen también de la ordenación de ramas: no son una tasa de error.
3. Correlacionamos sus vectores de distancias **cofenéticas**: ahora la referencia es otro árbol, no la distancia original.

**API:** `tanglegram(izquierdo, derecho)` recibe dos matrices de fusiones sobre las mismas observaciones y dibuja; no alinea especies ni selecciona un corte.

**Lectura ejecutada:** completo y promedio tienen correlación entre alturas cofenéticas **0,620**. Esto difiere de la fidelidad de completo frente a `d` (**0,727**). Dos jerarquías pueden parecerse entre sí sin representar bien la geometría original; tampoco esta correlación equivale a ARI de un corte.

In [ ]:
# Dos enlaces sobre las MISMAS distancias originales; no se consultan especies.
hc1 = linkage(d, method="complete")
hc2 = linkage(d, method="average")


def tanglegram(izquierdo, derecho):
    # 1. Dibuja las dos jerarquías mirando hacia el panel central.
    fig, axes = plt.subplots(1, 3, figsize=(14, 7), gridspec_kw={"width_ratios": [2, 1, 2]})
    a = dendrogram(izquierdo, orientation="left", no_labels=True, ax=axes[0],
                   color_threshold=0, above_threshold_color=COLOR_BASE)
    b = dendrogram(derecho, orientation="right", no_labels=True, ax=axes[2],
                   color_threshold=0, above_threshold_color=COLOR_BASE)
    # 2. Localiza cada flor en ambos órdenes y une sus posiciones; no une grupos o clases.
    posiciones_a = {observacion: 5 + 10 * i for i, observacion in enumerate(a["leaves"])}
    posiciones_b = {observacion: 5 + 10 * i for i, observacion in enumerate(b["leaves"])}
    for observacion in posiciones_a:
        axes[1].plot([0, 1], [posiciones_a[observacion], posiciones_b[observacion]],
                     color=COLOR_ACENTO, alpha=0.3, linewidth=0.6)
    # 3. Usa el mismo rango de posiciones para que cada conexión corresponda a la misma fila.
    for ax in axes:
        ax.set_ylim(0, 10 * len(a["leaves"]))
    axes[1].set(xlim=(0, 1))
    axes[1].axis("off")
    axes[0].set_title("Enlace completo")
    axes[2].set_title("Enlace promedio")
    fig.tight_layout()
    plt.show()


tanglegram(hc1, hc2)  # Los cruces también cambian si se permuta el orden de las ramas.

In [ ]:
# Cada columna recoge alturas cofenéticas de los mismos pares de flores.
dend_list = {"Completo": hc1, "Promedio": hc2}
pd.DataFrame({nombre: cophenet(arbol) for nombre, arbol in dend_list.items()}).corr()
# Esta correlación compara DOS ÁRBOLES; no compara cada árbol con d.

#### Cuatro enlaces sobre la misma geometría: completo, promedio, Ward y simple

Ward utiliza aquí `d`, que es **euclídeo**, por lo que su interpretación de incremento de inercia es válida. El enlace simple favorece cadenas por proximidad local; el completo controla el par más lejano de cada unión.

**Lectura de la matriz:** promedio y Ward dan **0,989**, mientras que completo y Ward dan **0,610**. Es una similitud lineal entre alturas, no identidad de árboles, coincidencia de cortes ni evidencia de especies. La diagonal vale 1 por comparar cada vector consigo mismo. Los sectores circulares de la figura representan el mismo coeficiente que el número, no porcentajes correctos.

In [ ]:
hc3 = linkage(d, method="ward")  # Ward válido: d proviene de distancias euclídeas.
hc4 = linkage(d, method="single")  # Enlace simple: mínimo de distancias entre los dos grupos.
dend_list.update({"Ward": hc3, "Simple": hc4})
# Compara alturas de primera unión de cada par entre los cuatro árboles.
cor_dend = pd.DataFrame({nombre: cophenet(arbol) for nombre, arbol in dend_list.items()}).corr()
cor_dend

In [ ]:
# Representa el triángulo inferior de cor_dend; no recalcula ni evalúa particiones.
fig, ax = plt.subplots(figsize=(6, 5))
for i in range(len(cor_dend)):
    for j in range(i+1):
        valor = cor_dend.iloc[i, j]
        _ = ax.add_patch(Circle((j, i), 0.42, edgecolor="#DCE4E8", facecolor="white"))
        # El ángulo del sector codifica |correlación| y el color su signo.
        _ = ax.add_patch(Wedge((j, i), 0.42, 90, 90 + 360 * abs(valor),
                               facecolor=COLOR_BASE if valor >= 0 else COLOR_ACENTO))
        _ = ax.text(j, i, f"{valor:.2f}", ha="center", va="center", fontsize=9)
_ = ax.set(xticks=range(4), yticks=range(4), xlim=(-0.6, 3.6), ylim=(3.6, -0.6),
           title="Correlaciones entre alturas cofenéticas de árboles")
_ = ax.set_xticklabels(cor_dend.columns, rotation=45)
_ = ax.set_yticklabels(cor_dend.index)
_ = ax.set_aspect("equal")
fig.tight_layout()
plt.show()

#### Micropráctica 2 — cambiar enlace y leer tres criterios distintos

Completa `enlace_practica` con el enlace **promedio**, ya mostrado. Mantén las mismas medidas, la misma distancia euclídea y $k=3$. Reutiliza `linkage`, `cophenet`, `cut_tree`, `resumen_particion` y ARI para comparar **fidelidad a `d`**, **reducción de dispersión** y **acuerdo con `ct`**.

Antes de ejecutar, explica por qué una correlación cofenética mayor no obliga a que ARI sea 1 ni a que el corte tenga menor WSS. No uses `Species` para decidir cuál conservar; los tres criterios responden a preguntas distintas.

In [ ]:
# Micropráctica opcional: las variables nuevas no sustituyen hc ni ct.
enlace_practica = ...  # Completa con el nombre del enlace promedio en SciPy.
if enlace_practica is Ellipsis:
    print("Micropráctica pendiente: completa el enlace y distingue los tres criterios.")
else:
    arbol_practica = linkage(d, method=enlace_practica)
    grupos_practica = renumerar(cut_tree(arbol_practica, n_clusters=3).ravel())
    resumen_practica = resumen_particion(iris2, grupos_practica)
    display(pd.Series({"Fidelidad cofenética frente a d": cophenet(arbol_practica, d)[0],
                       "WSS (cm²)": resumen_practica.inertia_,
                       "B/T (no acierto)": resumen_practica.betweenss / resumen_practica.totss,
                       "ARI frente al corte completo ct": adjusted_rand_score(ct, grupos_practica)}))
    # Redacta qué cambió: representar distancias, dispersión interna y pertenencia no son lo mismo.

## 5.3. Tendencia al agrupamiento: Hopkins (diapositiva 19)

La pregunta es si la nube se aparta de una **referencia uniforme**, no qué algoritmo gana ni cuántos grupos hay.

1. Muestreamos observaciones reales y medimos su distancia $w_i$ al vecino real más próximo, **excluyendo la propia fila**. Otras flores con medidas duplicadas pueden dar distancia cero.
2. Generamos puntos uniformes en la **caja de mínimos y máximos de cada variable**, no en la distribución conjunta real.
3. Medimos la distancia $u_i$ de cada punto uniforme a la observación real más próxima.
4. Comparamos las sumas de distancias euclídeas, sin elevarlas a la dimensión. Esta es la implementación didáctica conservada; hay variantes de Hopkins con otras potencias y referencias.

$$H_0=\frac{\sum_i w_i}{\sum_i w_i+\sum_i u_i}.$$

- **Cerca de 0:** vecinos reales próximos respecto a puntos uniformes; tendencia a concentración frente a esa referencia.
- **Cerca de 0,5:** compatible orientativamente con uniformidad.
- **Cerca de 1:** posible regularidad o repulsión.

**Orientación:** la diapositiva usa $1-H_0$, que se acerca a 1 cuando hay concentración. No confundas ambas convenciones. `n` indica cuántas filas se muestrean, **no** el número de grupos. El valor depende de escala, variables, semilla, tamaño muestral y caja uniforme. Asociación, correlaciones o límites del soporte también pueden alejarlo de 0,5: no es prueba de categorías naturales, elección de $k$, significación automática ni causalidad.

In [ ]:
def hopkins(datos, n=None, random_state=12345):
    # 1. Muestra filas reales sin reemplazo; n es tamaño muestral, NO número de grupos.
    a = np.asarray(datos, dtype=float)
    m = len(a) - 1 if n is None else int(n)
    if not 1 <= m < len(a):
        raise ValueError("El tamaño de muestra debe estar entre 1 y n-1.")
    rng = np.random.default_rng(random_state)
    indices = rng.choice(len(a), m, replace=False)
    # 2. Vecino real más próximo: excluye solo la propia fila; conserva posibles duplicados.
    reales = cdist(a[indices], a)
    reales[np.arange(m), indices] = np.inf
    w = reales.min(axis=1)
    # 3. Referencia uniforme independiente por variable dentro de su caja de mínimos/máximos.
    uniformes = rng.uniform(a.min(axis=0), a.max(axis=0), size=(m, a.shape[1]))
    u = cdist(uniformes, a).min(axis=1)
    # 4. Convención de este cuaderno: valores bajos sugieren concentración frente a la caja.
    return w.sum() / (w.sum() + u.sum())

In [ ]:
# Referente reproducible: 149 filas muestreadas, cuatro medidas sin escalar, semilla original.
hop = hopkins(iris2, n=len(iris2)-1, random_state=SEED)
hop
print(f"Convención H0: {hop:.3f}; convención complementaria de la diapositiva: {1-hop:.3f}")

**Lectura ejecutada:** Iris original da **$H_0=0,168$**, equivalente a **$1-H_0=0,832$**. Las flores reales tienen vecinos relativamente próximos frente a puntos uniformes en la caja. La referencia ignora las correlaciones entre medidas: esta concentración no demuestra tres grupos ni que sean especies.

#### Micropráctica 3 — referencia, escala y orientación

Completa el tamaño muestral con **30** y calcula Hopkins sobre Iris original y estandarizado, manteniendo la semilla. Expresa ambos valores en las dos convenciones. Compara el cambio con el referente de 149 filas, pero no busques el menor Hopkins como si fuese un procedimiento para elegir $k$.

In [ ]:
# Micropráctica opcional: usa la función ya definida sin tocar hop ni sus datos.
m_hop_practica = ...  # Completa con 30: número de filas muestreadas, no número de clústeres.
if m_hop_practica is Ellipsis:
    print("Micropráctica pendiente: completa la muestra e interpreta la referencia uniforme.")
else:
    hop_practica = pd.Series({"Original": hopkins(iris2, n=m_hop_practica, random_state=SEED),
                              "Estandarizado": hopkins(iris2_escalado, n=m_hop_practica, random_state=SEED)})
    display(pd.DataFrame({"H0 (concentración hacia 0)": hop_practica,
                          "1-H0 (concentración hacia 1)": 1-hop_practica}).round(3))
    # Explica por qué esta comparación no elige k ni identifica una causa de concentración.

## 5.4. Ejercicio 5-1 — Vinos (solución completa guiada)

**Intenta cada apartado antes de ejecutar su solución.** Conservamos la resolución completa; la micropráctica final permite variar una decisión sin sustituirla.

1. **Lectura:** inspecciona las 13 medidas de 178 vinos. Sus escalas y unidades son heterogéneas; comparar rangos originales de variables diferentes no mide su capacidad para separar grupos.
2. **Preparación:** estandariza cada columna y dibuja un mapa de calor.
3. **Árboles:** compara euclídea/Manhattan y completo/Ward. **Ward con euclídea es el único Ward con interpretación de aumento de inercia.** Se conserva el cuarto cálculo Manhattan/Ward como demostración de actualización de disimilitudes no euclídeas, explícitamente marcado; no lo presentes como Ward euclídeo válido.
4. **Particiones:** fija $k=3$ como pide el ejercicio y compara tamaños y ARI entre los cuatro resultados **sin mirar `Type` para escogerlos**.
5. **Perfiles:** describe el referente euclídea/completo mediante $B/T$, medias estandarizadas y rangos entre medias de grupo. Solo después cruza sus etiquetas con los tipos conocidos.

El primer fichero contiene medidas; el segundo añade `Type`. Comprobamos que las medidas coinciden fila a fila antes del contraste externo. `Type` queda separado: no entra en escalado, distancias, ajuste ni selección.

### 5.4.1. Localizar datos, revisar escalas y estandarizar

`DATA_PATH` se busca desde la carpeta actual hacia sus padres, para que funcione tanto al abrir la raíz como la carpeta de sesión. `pd.read_csv` usa comas en estos dos ficheros. El mapa de calor reordena según proximidad **estandarizada**; sus colores indican desviaciones respecto a la media de cada variable, no tipos de vino ni aciertos.

In [ ]:
# Busca la carpeta de datos sin depender de que el kernel arranque en la raíz del proyecto.
for carpeta in (Path.cwd(), *Path.cwd().parents):
    if (carpeta / "datos" / "wine2.txt").is_file():
        DATA_PATH = carpeta / "datos"
        break  # Conserva el primer directorio válido; no cambia la carpeta de trabajo.
else:
    raise FileNotFoundError("Abre el notebook dentro de la carpeta LaSalle_Estadistica.")

In [ ]:
# 1. Lee solo medidas para ajustar; reserva Type y comprueba su correspondencia fila a fila.
vinos = pd.read_csv(DATA_PATH / "wine2.txt")
vinos_con_tipo = pd.read_csv(DATA_PATH / "wine.data.txt")
tipos_vino = vinos_con_tipo["Type"]  # Solo evaluación externa, nunca una variable del ajuste.
assert len(vinos) == len(tipos_vino) and not vinos.isna().any().any()
assert vinos.shape == (178, 13) and np.allclose(vinos.to_numpy(), vinos_con_tipo.iloc[:, 1:].to_numpy())
# Los nombres difieren ligeramente entre ficheros; el orden y los valores de las medidas coinciden.
display(vinos.describe().T[["mean", "std", "min", "max"]].round(2))
# 2. Da peso comparable a las desviaciones de cada variable; conserva vinos en su escala original.
vinos_escalados = escalar(vinos)
mapa_vino = sns.clustermap(vinos_escalados, method="complete", metric="euclidean",
                            cmap="vlag", figsize=(11, 7), yticklabels=False)
_ = mapa_vino.figure.suptitle("Vinos: cada variable en unidades de desviación típica", y=1.02)
plt.show()  # El mapa no usa Type para ordenar filas o escoger parámetros.

**Lectura de la preparación:** las 178 filas conservan 13 medidas válidas y coinciden con las del fichero que añade `Type`. El resumen original muestra magnitudes muy diferentes, por ejemplo `Proline` frente a `Hue`; tras escalar se comparan desviaciones típicas, no importes o concentraciones intercambiables. Estandarizar no garantiza mejor estructura ni acuerdo externo.

### 5.4.2. Cuatro árboles y cuatro cortes con $k=3$

La solución reutiliza `pdist`, `linkage` y `cut_tree`. Se guardan los cuatro árboles **y** las etiquetas para compararlos después. La correlación de cada árbol se calcula con su propia métrica: una cifra mayor tras cambiar métrica no demuestra una mejor tipología.

**Lectura ejecutada:** los tamaños son **69/58/51** (euclídea/completo), **64/58/56** (euclídea/Ward), **97/29/52** (Manhattan/completo) y **63/62/53** (actualización Manhattan/Ward). El cambio de pertenencias importa más que el orden o el color de las hojas. Sus correlaciones cofenéticas son aproximadamente **0,592; 0,662; 0,638; 0,698**, respectivamente. La última **no** legitima Ward no euclídeo ni sus alturas como incremento de WSS.

In [ ]:
# Solución completa: conserva los cuatro cruces y sus cortes, sin utilizar Type.
arboles_vino = {}
filas_vino = []
for metrica in ("euclidean", "cityblock"):
    distancia_vino = pdist(vinos_escalados, metric=metrica)
    for enlace in ("complete", "ward"):
        # Manhattan/Ward se conserva solo como actualización NO euclídea, no Ward de inercia.
        arbol_vino = linkage(distancia_vino, method=enlace)
        grupos_vino = renumerar(cut_tree(arbol_vino, n_clusters=3).ravel())
        nombre = f"{metrica} / {enlace}"
        arboles_vino[nombre] = (arbol_vino, grupos_vino)
        filas_vino.append({"Combinación": nombre, "Cophenética": cophenet(arbol_vino, distancia_vino)[0],
                           "Tamaños (k=3)": tuple(np.bincount(grupos_vino)),
                           "Ward euclídeo": "No: actualización de disimilitudes" if enlace == "ward" and metrica != "euclidean"
                                            else ("Sí" if enlace == "ward" else "No aplica")})
display(pd.DataFrame(filas_vino).set_index("Combinación").round(3))
# Dibuja cada árbol: alturas definidas por SU combinación, no una escala común de WSS.
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, (nombre, (arbol_vino, _)) in zip(axes.flat, arboles_vino.items()):
    _ = dendrogram(arbol_vino, no_labels=True, ax=ax, color_threshold=0,
                   above_threshold_color=COLOR_BASE)
    advertencia = " · Ward no euclídeo" if nombre == "cityblock / ward" else ""
    _ = ax.set(title=nombre + advertencia, xlabel="Vinos ordenados", ylabel="Altura de la fusión")
fig.tight_layout()
plt.show()

### 5.4.3. Comparar pertenencias, interpretar perfiles y mirar tipos al final

1. Usamos euclídea/completo como **referente del ejercicio**, no por maximizar acuerdo con tipos.
2. ARI compara cada partición con ese referente: **1,000; 0,603; 0,432; 0,581**. Ninguno de los tres árboles alternativos conserva exactamente sus grupos.
3. `resumen_particion` calcula WSS y $B/T$ sobre los vinos **estandarizados**. El referente reduce **35,9 %** de la dispersión, no clasifica correctamente el 35,9 % de vinos.
4. `groupby(...).mean()` resume por **grupos descubiertos**. El rango máximo menos mínimo de sus medias estandarizadas es menor en **`Ash` (0,46 desviaciones típicas)** y después `Magnesium` (0,78); aquí `Flavanoids` muestra un contraste mayor (1,86).
5. La tabla de `Type` se construye al final: el primer grupo mezcla 51 vinos de tipo 1 y 18 de tipo 2; el segundo mezcla 8 y 50, y el tercero reúne 3 de tipo 2 y 48 de tipo 3. Son grupos descubiertos con mezcla, **no nombres de tipos recuperados automáticamente**.

**Cautela de perfiles:** una diferencia pequeña entre medias de grupo no implica que una variable sea inútil en todas las geometrías; las medias no muestran toda la dispersión interna. Los índices, rangos y tabla externa describen esta solución, no una causa de la tipología.

In [ ]:
# 1. Referente fijado por el ejercicio; ARI compara particiones, no aciertos de tipos.
referencia_vino = arboles_vino["euclidean / complete"][1]
comparacion_vino = pd.Series({nombre: adjusted_rand_score(referencia_vino, grupos)
                              for nombre, (_, grupos) in arboles_vino.items()},
                             name="Rand ajustado frente a euclídea/completo")
display(comparacion_vino.round(3))
# 2. WSS y B/T se calculan con las 13 variables estandarizadas de ESTA partición.
resumen_vino = resumen_particion(vinos_escalados, referencia_vino)
print(f"Reducción relativa de dispersión B/T (k=3, no acierto): "
      f"{resumen_vino.betweenss / resumen_vino.totss:.1%}")
# 3. Describe los grupos descubiertos, nunca los tipos, mediante medias y rangos comparables.
perfiles_vino = vinos_escalados.groupby(referencia_vino).mean()
perfiles_vino.index.name = "Grupo"
display(perfiles_vino.round(2))
display((perfiles_vino.max() - perfiles_vino.min()).sort_values().rename(
    "Rango entre medias de grupo (en desviaciones típicas)").round(2))
# 4. Solo DESPUÉS de definir y describir la partición: inspección externa, no selección.
display(pd.crosstab(tipos_vino, referencia_vino + 1, rownames=["Tipo conocido"],
                    colnames=["Grupo descubierto"]))

#### Micropráctica 4 — otra resolución del mismo referente de vinos

Completa $k$ con **4** y corta el árbol euclídea/completo ya guardado. Resume tamaños y $B/T$ con `vinos_escalados`; usa ARI para comparar pertenencias con el referente en tres grupos. Explica por qué un $B/T$ mayor al refinar no valida cuatro tipos conocidos.

No borres ni completes huecos en la solución anterior: está resuelta. Esta variante utiliza variables nuevas y **no consulta `Type`** para elegir el corte.

In [ ]:
# Micropráctica opcional: conserva toda la solución 5-1 y no usa los tipos conocidos.
k_vino_practica = ...  # Completa con 4.
if k_vino_practica is Ellipsis:
    print("Micropráctica pendiente: completa el corte y distingue dispersión de tipos conocidos.")
else:
    grupos_vino_practica = renumerar(cut_tree(arboles_vino["euclidean / complete"][0],
                                             n_clusters=k_vino_practica).ravel())
    resumen_vino_practica = resumen_particion(vinos_escalados, grupos_vino_practica)
    display(pd.Series(resumen_vino_practica.size, index=range(1, k_vino_practica+1), name="Tamaño"))
    print(f"B/T (no acierto): {resumen_vino_practica.betweenss / resumen_vino_practica.totss:.1%}")
    print(f"ARI frente a k=3: {adjusted_rand_score(referencia_vino, grupos_vino_practica):.3f}")
    # Argumenta por qué una subdivisión más fina no es una clase conocida adicional.

## 5.5. K-means: centros, inicios y resolución (diapositivas 23–31)

Para un $k$ elegido, k-means busca poca **inercia intra euclídea**:

$$\min_{C_1,\ldots,C_k}\sum_{g=1}^{k}\sum_{i\in C_g}\lVert x_i-\bar{x}_g\rVert^2.$$

1. **Lloyd** alterna asignar al centro más próximo y recalcular medias.
2. La implementación didáctica **Hartigan–Wong** ensaya transferencias que reduzcan el mismo coste.
3. Ambos pueden acabar en un **mínimo local**. `n_init` compara inicios y conserva la menor WSS para **el mismo $k$, datos y escala**, no la mayor silhouette ni coincidencia con especies.
4. Una semilla hace reproducible el experimento; no certifica el mínimo global ni identidad con R u otra biblioteca.

Los centroides son medias, no necesariamente observaciones reales. Los atípicos pueden arrastrarlos; las asignaciones al centro más próximo favorecen regiones convexas y no identifican bien cualquier geometría.

### 5.5.1. Un inicio sobre las medidas originales de Iris

`ajustar_kmeans(iris2, k=3, n_init=1, random_state=SEED)` utiliza por defecto Hartigan–Wong **sin escalar**. `labels_` identifica el grupo de cada flor; `cluster_centers_` contiene cuatro medias en cm por grupo; `n_iter_` cuenta iteraciones, no exactitud. Los otros campos resumen tamaños y sumas de cuadrados.

**Lectura ejecutada:** el inicio conservado da tamaños **50/62/38**, $T=681,371$ y $W=78,851$ cm². Los tres centros representan perfiles medios de medidas, no especies asignadas. Comprueba que las WSS parciales suman la WSS total y que $T=W+B$.

In [ ]:
# Un único inicio Hartigan–Wong: cuatro medidas originales; Species queda fuera.
km0 = ajustar_kmeans(iris2, k=3, n_init=1, random_state=SEED)
pd.Series(km0.labels_ + 1, index=iris2.index, name="Grupo")  # Una etiqueta arbitraria por flor.

In [ ]:
# Centroides = medias de las flores de cada grupo en las cuatro variables originales (cm).
pd.DataFrame(km0.cluster_centers_, columns=iris2.columns, index=np.arange(1, 4))
# No tienen que coincidir con una flor observada ni con la media de una especie conocida.

In [ ]:
# Resume la partición con LOS MISMOS datos y escala usados para ajustar.
resumen_km0 = resumen_particion(iris2, km0.labels_)
resumen_km0.totss       # T: distancias cuadráticas a la media global (cm²).
resumen_km0.withinss    # WSS de cada grupo respecto a su propio centro (cm²).
resumen_km0.inertia_    # W: suma de las WSS parciales (cm²), objetivo del ajuste.
resumen_km0.betweenss   # B = T - W: dispersión entre centros ponderada por tamaños (cm²).
resumen_km0.size        # Número de flores en cada grupo, no especies recuperadas.
km0.n_iter_            # Iteraciones de transferencias de este inicio.
assert np.isclose(resumen_km0.withinss.sum(), resumen_km0.inertia_)
assert np.isclose(resumen_km0.totss, resumen_km0.inertia_ + resumen_km0.betweenss)

#### Leer $B/T$: reducción de dispersión, no acierto

El cociente $B/T=1-W/T$ es adimensional. En este ajuste es aproximadamente **88,4 %**: tres centros reducen esa proporción de dispersión respecto al centro global. No se han usado especies, así que no puede interpretarse como porcentaje de flores correctamente clasificadas.

Renumerar grupos no cambia centros como conjunto, WSS ni $B/T$; solo el orden de los resultados. Un valor mayor puede deberse a añadir grupos, incluso sin segmentación útil.

In [ ]:
resumen_km0.betweenss / resumen_km0.totss  # Proporción de dispersión reducida, NO acierto externo.

### 5.5.2. Número de grupos: dos formas de leer el mismo codo (diapositiva 28)

1. Ajustamos $k=1,\ldots,10$ con **diez inicios** por valor de $k$ y la semilla original.
2. Dibujamos $B_k/T$ y después $W_k$: expresan el mismo cambio porque $B_k/T=1-W_k/T$.
3. Buscamos reducciones grandes seguidas de mejoras menores; no el máximo de $B/T$ por sí solo.

El mínimo teórico de WSS no aumenta al añadir grupos; ajustes concretos pueden incumplirlo si terminan en mínimos locales. En la curva ejecutada las primeras WSS son **681,371; 152,348; 78,851; 57,228** cm². Pasar de dos a tres mejora el criterio, pero cada grupo adicional aporta menos en esta zona.

**Lectura frente al árbol:** con $k=3$, k-means da WSS **78,851**, menor que **89,525** del corte completo, en la misma escala. Es esperable que optimizar centroides reduzca más WSS que cortar un árbol con otro criterio; no implica mejor fidelidad cofenética ni mayor exactitud externa.

#### Codo como proporción de dispersión reducida

No compares WSS absolutas de Iris original con vinos estandarizados, HCPC o gastos transformados. Tampoco uses `Species` para elegir $k$.

In [ ]:
modelos_k = {k: ajustar_kmeans(iris2, k=k, n_init=10, random_state=SEED) for k in range(1, 11)}
VE = pd.Series({k: resumen_particion(iris2, modelos_k[k].labels_).betweenss / resumen_km0.totss
                for k in range(2, 11)}, name="Variabilidad explicada")
ax = VE.plot(marker="o", color=COLOR_BASE)
_ = ax.set(xlabel="Número de grupos", ylabel="Variabilidad explicada", title="Regla del codo")
_ = estilo_ejes(ax, cuadricula=True)
plt.show()

#### Regla del codo: inercia dentro de los grupos

In [ ]:
inercia_k = pd.Series({k: modelo.inertia_ for k, modelo in modelos_k.items()})
ax = inercia_k.plot(marker="o", color=COLOR_BASE)
_ = ax.set(xlabel="Número de grupos", ylabel="Inercia intra", title="Regla del codo")
_ = estilo_ejes(ax, cuadricula=True)
plt.show()

#### Número de grupos según los indicadores internos

In [ ]:
ncluster = evaluar_indices(iris2, metodo="kmeans", min_k=2, max_k=15, random_state=SEED)
ncluster.Best_n

In [ ]:
ax = ncluster.Votes.plot.bar(rot=0, color=COLOR_BASE)
_ = ax.set(xlabel="Número de grupos", ylabel="Votos", title="Recomendaciones de los indicadores")
_ = estilo_ejes(ax, cuadricula=True)
plt.show()

In [ ]:
indices_escalados = escalar(ncluster.All_index)
fig, ax = plt.subplots(figsize=(15, 6))
_ = sns.heatmap(indices_escalados, cmap="vlag", center=0, ax=ax)
_ = ax.set(title="Indicadores estandarizados", xlabel="Indicador", ylabel="Número de grupos")
fig.tight_layout()
plt.show()

### Representación: tres grupos

In [ ]:
km3 = ajustar_kmeans(iris2, k=3, n_init=10, random_state=SEED)
pares_coloreados(iris2, km3.labels_)

In [ ]:
pr_comp = PCA().fit(iris2)
coordenadas = pr_comp.transform(iris2)
x, y = coordenadas[:, 0], coordenadas[:, 1]
fig, ax = plt.subplots()
_ = ax.scatter(x, y, c=km3.labels_, cmap=MAPA_GRUPOS, vmin=-0.5, vmax=9.5, s=24)
_ = ax.set(xlabel="CP1", ylabel="CP2", title="K-means: tres grupos")
fig.tight_layout()
plt.show()

In [ ]:
representar_clusters(iris2, km3.labels_, titulo="K-means: tres grupos y envolventes convexas")

#### Tres dimensiones: color por grupo y símbolo por especie. Cambia el ángulo de visión.

In [ ]:
def iris_3d(angulo):
    fig = plt.figure(figsize=(9, 6))
    ax = fig.add_subplot(111, projection="3d")
    for especie, marcador in enumerate(["o", "s", "^"]):
        mascara = dataset.target == especie
        ax.scatter(iris2.loc[mascara, "Sepal.Length"], iris2.loc[mascara, "Sepal.Width"],
                   iris2.loc[mascara, "Petal.Length"], c=km3.labels_[mascara], cmap=MAPA_GRUPOS,
                   vmin=-0.5, vmax=9.5, marker=marcador, label=dataset.target_names[especie])
    for fila, grupo in zip(iris2.to_numpy(), km3.labels_):
        ax.plot([fila[0], fila[0]], [fila[1], fila[1]], [0, fila[2]],
                color=COLORES_GRUPO[int(grupo) % 10], alpha=0.15, linewidth=0.5)
    ax.set(xlabel="Sepal.Length", ylabel="Sepal.Width", zlabel="Petal.Length")
    ax.view_init(elev=20, azim=angulo)
    ax.legend(frameon=False)
    fig.tight_layout()
    display(widgets.HTML(value=imagen_control(fig, f"Iris en tres dimensiones, ángulo {angulo}°")))

In [ ]:
control_3d = widgets.interactive(
    iris_3d,
    angulo=widgets.IntSlider(value=60, min=0, max=180, step=5, description="Ángulo", continuous_update=False),
)
display(panel_interactivo("Gira la representación 3D de Iris", control_3d))

### Representación: dos grupos

In [ ]:
km2 = ajustar_kmeans(iris2, k=2, n_init=10, random_state=SEED)
pares_coloreados(iris2, km2.labels_)

In [ ]:
fig, ax = plt.subplots()
_ = ax.scatter(x, y, c=km2.labels_, cmap=MAPA_GRUPOS, vmin=-0.5, vmax=9.5, s=24)
_ = ax.set(xlabel="CP1", ylabel="CP2", title="K-means: dos grupos")
fig.tight_layout()
plt.show()

In [ ]:
representar_clusters(iris2, km2.labels_, titulo="K-means: dos grupos y envolventes convexas")

#### Explorar k, los inicios y el algoritmo

In [ ]:
def explorar_kmeans(k, inicios, algoritmo):
    modelo = ajustar_kmeans(iris2, k, n_init=inicios, algoritmo=algoritmo, random_state=SEED)
    grafico = representar_clusters(iris2, modelo.labels_, titulo=f"K-means: {k} grupos", mostrar=False)
    display(widgets.HTML(value=grafico))
    resumen = resumen_particion(iris2, modelo.labels_)
    display(pd.Series({"Inercia intra": modelo.inertia_, "Variabilidad explicada": resumen.betweenss / resumen.totss,
                       "Silhouette": silhouette_score(iris2, modelo.labels_)}))

In [ ]:
control_kmeans = widgets.interactive(
    explorar_kmeans,
    k=widgets.IntSlider(value=3, min=2, max=8, description="Grupos", continuous_update=False),
    inicios=widgets.SelectionSlider(options=[1, 5, 10], value=10, description="Inicios", continuous_update=False),
    algoritmo=widgets.Dropdown(options=[("Hartigan–Wong", "hartigan-wong"), ("Lloyd", "lloyd")], description="Algoritmo"),
)
display(panel_interactivo("Compara los ajustes de k-means", control_kmeans))

## 5.6. K-medoides (diapositivas 33–34)

Un **medoide es una observación real**, no una media que puede quedar fuera de la nube. PAM busca representantes que reduzcan la distancia total al medoide más próximo:

$$
\min_{m_1,\ldots,m_k\;\text{observaciones}}\;\sum_{i=1}^{n}\min_{g=1,\ldots,k}d(x_i,m_g).
$$

**BUILD** elige los representantes iniciales y **SWAP** prueba intercambios hasta que ninguno mejora el coste. El código usa distancia euclídea, salvo que se suministre otra matriz de disimilitudes. Aunque el resultado guarda el coste en `inertia_`, aquí es **suma de distancias a medoides**, no WSS (suma de distancias **al cuadrado** a medias) de k-means: sus valores no son comparables directamente.

**CLARA**, implementado en 5.8, aplica PAM a muestras y evalúa sus representantes en todos los datos. Reduce cálculo, pero puede omitir grupos pequeños. Usar observaciones reales puede reducir la sensibilidad a atípicos; no garantiza resistencia a cualquier contaminación.

In [ ]:
def pam(datos, k, distancias=None, max_iter=100):
    # Preparación: una fila por observación; D contiene disimilitudes, no cuadrados.
    a = np.asarray(datos, dtype=float)
    D = squareform(pdist(a)) if distancias is None else np.asarray(distancias)
    if not 1 <= k < len(a):
        raise ValueError("k debe ser menor que el número de observaciones.")
    # BUILD: añadir observaciones reales que reduzcan la distancia total.
    medoides = [int(D.sum(axis=0).argmin())]
    while len(medoides) < k:
        cercana = D[:, medoides].min(axis=1)
        costes = np.minimum(D, cercana[:, None]).sum(axis=0)
        costes[medoides] = np.inf
        medoides.append(int(costes.argmin()))
    # SWAP: buscar el mejor intercambio de un medoide por otra observación.
    historial = []
    for iteracion in range(max_iter):
        actuales = D[:, medoides]
        asignacion = actuales.argmin(axis=1)
        primera = actuales.min(axis=1)
        segunda = np.partition(actuales, 1, axis=1)[:, 1] if k > 1 else np.full(len(a), np.inf)
        coste_actual = primera.sum()
        historial.append(coste_actual)
        mejor = (coste_actual, None, None)
        # Si retiramos el medoide j, sus miembros recurren al segundo más próximo.
        for j in range(k):
            base = np.where(asignacion == j, segunda, primera)
            costes = np.minimum(D, base[:, None]).sum(axis=0)
            costes[medoides] = np.inf
            candidato = int(costes.argmin())
            if costes[candidato] < mejor[0] - 1e-9:
                mejor = (costes[candidato], j, candidato)
        if mejor[1] is None:
            break
        medoides[mejor[1]] = mejor[2]
    # Resultado: asignar al medoide más cercano; inertia_ es coste PAM, no WSS.
    etiquetas = D[:, medoides].argmin(axis=1)
    return SimpleNamespace(labels_=etiquetas, medoid_indices_=np.array(medoides),
                           cluster_centers_=a[medoides], inertia_=D[:, medoides].min(axis=1).sum(),
                           history_=np.array(historial), n_iter_=iteracion + 1)

In [ ]:
# Ajuste fijo: los mismos datos de Iris y tres representantes observados.
kmedoids3 = pam(iris2, k=3)
# Proyección para comparar la partición con la figura de k-means, no para ajustar.
representar_clusters(iris2, kmedoids3.labels_, titulo="PAM: tres grupos")

### Concordancia de k-means y PAM

El **índice de Rand ajustado (ARI)** compara qué pares de observaciones quedan juntos, descontando el acuerdo esperado por azar. Vale 1 para particiones idénticas **aunque se permuten sus etiquetas**, suele rondar 0 para un acuerdo aleatorio y puede ser negativo. Compara particiones: **no es exactitud de clasificación**.

In [ ]:
# Acuerdo entre dos ajustes no supervisados; no se usan las especies para ajustar.
adjusted_rand_score(km3.labels_, kmedoids3.labels_)

### Concordancia con las especies de Iris

Ahora `Species` aporta una referencia externa. Un ARI alto indica asociación con esas especies, no que los algoritmos las hayan utilizado ni que cada flor esté «bien clasificada».

In [ ]:
# Evaluación externa posterior: Species no intervino en los centros de k-means.
adjusted_rand_score(km3.labels_, iris["Species"])

### Lectura de 5.6 y microprácticas

**Resultado confirmado en el kernel:** PAM y k-means dan la misma partición de Iris (ARI = **1**); frente a `Species`, el ARI de k-means es **0.730**. El primero no significa «100 % de aciertos»: solo acuerdo entre métodos. Los medoides tienen índices de fila **79, 8 y 113** en `iris2` (no números de especie). El coste PAM es **98.13** y la WSS de k-means **78.85**; la diferencia numérica no permite decidir cuál es mejor.

**Microprácticas opcionales, agrupadas y sin dependencias posteriores:**
1. Localiza esas filas en `iris2` y explica qué diferencia a un medoide de la media de su grupo.
2. Razona por qué renombrar los grupos no cambia el ARI y por qué ARI = 1 no exige centros iguales.
3. Explica qué podría perder CLARA si un grupo pequeño faltase de todas sus muestras.

Responde aparte o en una celda propia: ningún resultado de estas prácticas alimenta los ejemplos siguientes.

## 5.7. Grupos no convexos: anillos y kernel

Simulamos **500 puntos en un disco** de radio entre 0 y 1 y **500 en una corona** de radios entre 2 y 3. Radio y ángulo se extraen de distribuciones uniformes, con la semilla habitual.

**Radio uniforme no significa área uniforme:** intervalos radiales iguales reciben probabilidades iguales, aunque los más externos abarcan más superficie. Por eso la densidad por unidad de área es mayor cerca del centro del disco y del borde interior de la corona. Conservamos esta simulación, no la sustituimos por otra.

Ajustaremos sobre `df_anillos_escalado`, pero dibujaremos las coordenadas originales con la misma escala en ambos ejes. Así distinguimos la geometría del ajuste de la de su representación.

In [ ]:
# Simulación reproducible: radios uniformes, no puntos uniformes por unidad de área.
n = 500
rng = np.random.default_rng(SEED)
# Disco interior: coordenadas polares convertidas a cartesianas.
r1 = rng.uniform(0, 1, n)
alfa1 = rng.uniform(0, 2 * np.pi, n)
x1, y1 = r1 * np.cos(alfa1), r1 * np.sin(alfa1)
# Corona exterior: el hueco entre radios 1 y 2 separa los dos orígenes simulados.
r2 = rng.uniform(2, 3, n)
alfa2 = rng.uniform(0, 2 * np.pi, n)
x2, y2 = r2 * np.cos(alfa2), r2 * np.sin(alfa2)
# Datos originales para dibujar; versión estandarizada para ambos ajustes.
df_anillos = pd.DataFrame({"x": np.r_[x1, x2], "y": np.r_[y1, y2]})
df_anillos_escalado = escalar(df_anillos)

In [ ]:
# Vista de los datos sin etiquetas: la escala igual conserva la forma circular.
fig, ax = plt.subplots(figsize=(6, 6))
_ = ax.scatter(df_anillos["x"], df_anillos["y"], s=10, color=COLOR_BASE)
_ = ax.set(xlabel="x", ylabel="y", title="Disco y corona circular")
_ = ax.set_aspect("equal")
fig.tight_layout()
plt.show()

In [ ]:
# Ajuste euclídeo fijo sobre variables estandarizadas, con un inicio y la misma semilla.
km_anillos = ajustar_kmeans(df_anillos_escalado, k=2, n_init=1, random_state=SEED)
# Colorear las coordenadas originales no cambia los datos recibidos por el algoritmo.
fig, ax = plt.subplots(figsize=(6, 6))
_ = ax.scatter(df_anillos["x"], df_anillos["y"], c=km_anillos.labels_,
               cmap=MAPA_GRUPOS, vmin=-0.5, vmax=9.5, s=10)
_ = ax.set(title="K-means en grupos no convexos", xlabel="x", ylabel="y")
_ = ax.set_aspect("equal")
fig.tight_layout()
plt.show()

### K-means con kernel gaussiano

Una corona rodea al disco: con dos centros euclídeos, las regiones de asignación se separan por una recta, no por un anillo. Usamos similitudes gaussianas calculadas **en las coordenadas estandarizadas**:

$$
K(x_i,x_j)=\exp\!\left(-\gamma\lVert x_i-x_j\rVert^2\right),\qquad \gamma=\frac{1}{2\sigma^2}.
$$

Un $\gamma$ menor equivale a una anchura $\sigma$ mayor: incluso puntos lejanos resultan similares. Al aumentarlo, la similitud se vuelve más local; **más grande no implica mejor**, porque puede perder conexiones entre puntos del mismo grupo. Cambiar la escala de las variables también cambia esa geometría.

El código **inicializa con una representación espectral** de la matriz de similitudes y después actualiza distancias a centros en el espacio implícito del kernel. No es k-means sobre las coordenadas originales ni una ejecución puramente espectral. Su historial de costes pertenece a ese espacio: tampoco se compara numéricamente con WSS euclídea o coste PAM.

El ajuste fijo usa $\gamma=2$; el control permite explorar otros valores sin sobrescribir `kk`.

In [ ]:
def kernel_kmeans(datos, k=2, gamma=2.0, random_state=12345, max_iter=100):
    # Similitudes gaussianas: gamma actúa sobre las distancias de los datos recibidos.
    a = np.asarray(datos, dtype=float)
    K = np.exp(-gamma * squareform(pdist(a, metric="sqeuclidean")))
    # Inicialización espectral: normalizar afinidades y usar sus k autovectores superiores.
    pesos = np.sqrt(K.sum(axis=1))
    normalizada = K / np.outer(pesos, pesos)
    with threadpool_limits(limits=1):
        _, vectores = eigh(normalizada, subset_by_index=[len(a) - k, len(a) - 1])
        vectores = vectores / np.linalg.norm(vectores, axis=1, keepdims=True)
        etiquetas = KMeans(k, n_init=10, random_state=random_state).fit_predict(vectores)
    # Actualizaciones kernel: distancia cuadrática al centro implícito de cada grupo.
    historial = []
    for iteracion in range(max_iter):
        distancias = np.empty((len(a), k))
        for grupo in range(k):
            miembros = np.flatnonzero(etiquetas == grupo)
            if not len(miembros):
                raise ValueError("El kernel ha producido un grupo vacío; cambia gamma.")
            distancias[:, grupo] = np.diag(K) - 2 * K[:, miembros].mean(axis=1) + K[np.ix_(miembros, miembros)].mean()
        # Reasignar sin aceptar un grupo vacío; detenerse si las etiquetas no cambian.
        nuevas = distancias.argmin(axis=1)
        historial.append(float(distancias[np.arange(len(a)), etiquetas].sum()))
        if np.array_equal(nuevas, etiquetas):
            break
        if len(np.unique(nuevas)) < k:
            break
        etiquetas = nuevas
    # El historial mide el objetivo en el espacio del kernel, no la WSS original.
    return SimpleNamespace(labels_=etiquetas, history_=np.array(historial), n_iter_=iteracion + 1)

In [ ]:
# Resultado reproducible de referencia: no se modifica al mover el control.
kk = kernel_kmeans(df_anillos_escalado, k=2, gamma=2.0, random_state=SEED)

def explorar_kernel(gamma):
    # Cada callback ajusta un resultado local con la escala y la semilla fijas.
    resultado = kernel_kmeans(df_anillos_escalado, k=2, gamma=gamma, random_state=SEED)
    # Dibujar en coordenadas originales permite reconocer el disco y la corona.
    fig, ax = plt.subplots(figsize=(6, 6))
    _ = ax.scatter(df_anillos["x"], df_anillos["y"], c=resultado.labels_,
                   cmap=MAPA_GRUPOS, vmin=-0.5, vmax=9.5, s=10)
    titulo = f"K-means con kernel gaussiano: gamma = {gamma:.2f}"
    _ = ax.set(xlabel="x", ylabel="y", title=titulo)
    _ = ax.set_aspect("equal")
    fig.tight_layout()
    # La imagen HTML mantiene el gráfico integrado en el panel del widget.
    display(widgets.HTML(value=imagen_control(fig, titulo)))

In [ ]:
# Exploración opcional: recalcular solo al soltar el deslizador, sin alterar kk.
control_kernel = widgets.interactive(
    explorar_kernel,
    gamma=widgets.FloatSlider(value=2.0, min=0.5, max=4, step=0.25, description="Gamma", continuous_update=False),
)
display(panel_interactivo("Explora la anchura del kernel", control_kernel))

### Lectura de 5.7 y microprácticas

**En las figuras y el kernel actuales**, k-means mezcla los orígenes simulados: sus grupos contienen **420/80 puntos del disco** y **277/223 de la corona**, respectivamente. No recupera la separación radial. Con $\gamma=2$, el kernel coloca los **500 puntos de la corona en un grupo y los 500 del disco en el otro**; la inicialización ya resulta estable en la primera iteración. Es concordancia con un origen **simulado**, no garantía para cualquier nube, escala o $\gamma$.

**Microprácticas opcionales, sin dependencias:**
1. Compara en el control $\gamma=0.5$, $2$ y $4$: describe la forma de los grupos, no solo sus colores.
2. Explica por qué un radio uniforme no produce densidad uniforme sobre el área.
3. Razona qué cambia al estandarizar las coordenadas antes del kernel, aunque la figura use los ejes originales.

Registra las respuestas aparte; no sustituyas `kk` ni los datos fijos del ejemplo.

## 5.8. Valores atípicos

Simulamos **1000 observaciones de dos grupos base**, cerca de $(10,10)$ y $(100,10)$, y añadimos tres puntos alrededor de $(100\,000,100\,000)$. Comparamos k-means, PAM y CLARA con **dos grupos**, sobre los **datos originales, sin logaritmo ni estandarización**.

Los **ejes logarítmicos solo afectan a la figura**: permiten mostrar juntos los puntos próximos y los muy alejados. No cambian las distancias utilizadas al ajustar. K-means utiliza medias y distancia cuadrática; PAM utiliza observaciones reales y distancia sin cuadrar; CLARA aproxima el objetivo PAM mediante muestras.

No anticipes que PAM conservará los dos grupos base: un atípico suficientemente lejano también puede justificar un medoide propio. Contrasta esa posibilidad con el resultado obtenido.

In [ ]:
def clara(datos, k, samples=5, sampsize=None, random_state=12345):
    # Preparación: muestras reproducibles; por defecto, 40 + 2k observaciones.
    a = np.asarray(datos, dtype=float)
    rng = np.random.default_rng(random_state)
    tamano = min(len(a), 40 + 2 * k) if sampsize is None else min(len(a), sampsize)
    if tamano <= k:
        raise ValueError("El tamaño de muestra debe ser mayor que k.")
    mejor = None
    for _ in range(samples):
        # Conservar los mejores medoides previos y completar la muestra sin reemplazo.
        conservar = np.array([], dtype=int) if mejor is None else mejor.medoid_indices_
        candidatos = np.setdiff1d(np.arange(len(a)), conservar)
        indices = np.r_[conservar, rng.choice(candidatos, tamano - len(conservar), replace=False)]
        # PAM local: traducir los índices de la muestra a filas del conjunto completo.
        local = pam(a[indices], k)
        medoides = indices[local.medoid_indices_]
        # Evaluación global: elegir por suma de distancias a medoides, no por WSS.
        distancias = cdist(a, a[medoides])
        coste = distancias.min(axis=1).sum()
        if mejor is None or coste < mejor.inertia_:
            mejor = SimpleNamespace(labels_=distancias.argmin(axis=1), medoid_indices_=medoides,
                                     cluster_centers_=a[medoides], inertia_=coste)
    return mejor

In [ ]:
def generar_atipicos(nout=3):
    # Regenerar la simulación con la semilla fija para cada número de atípicos.
    rng = np.random.default_rng(SEED)
    n = 1000
    # Dos grupos se separan por x1; los atípicos se alejan en ambos ejes.
    x1 = np.r_[rng.normal(10, 1, n//2), rng.normal(100, 1, n//2), rng.normal(10**5, 1, nout)]
    x2 = np.r_[rng.normal(10, 1, n), rng.normal(10**5, 1, nout)]
    return pd.DataFrame({"x1": x1, "x2": x2})

# Comparación fija: mismas filas originales y k=2 para los tres métodos.
df_atipicos = generar_atipicos(nout=3)
km_atipicos = ajustar_kmeans(df_atipicos, k=2, n_init=1, random_state=SEED)
pam_atipicos = pam(df_atipicos, k=2)
clara_atipicos = clara(df_atipicos, k=2, random_state=SEED)

In [ ]:
def dibujar_atipicos(tabla, modelos, mostrar=True):
    # Cuatro paneles sobre las mismas observaciones, sin recalcular las particiones.
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    for ax, (nombre, grupos) in zip(axes.flat, modelos.items()):
        if grupos is None:
            _ = ax.scatter(tabla["x1"], tabla["x2"], color=COLOR_BASE, s=14)
        else:
            _ = ax.scatter(tabla["x1"], tabla["x2"], c=grupos,
                           cmap=MAPA_GRUPOS, vmin=-0.5, vmax=9.5, s=14)
        # Solo representación: poner los ejes en log no transforma el ajuste previo.
        _ = ax.set(xscale="log", yscale="log", xlabel="x1", ylabel="x2", title=nombre)
    fig.tight_layout()
    if not mostrar:
        return imagen_control(fig, "Comparación de datos, k-means, PAM y CLARA con valores atípicos")
    plt.show()

# Figura de referencia; el widget generará figuras locales por separado.
dibujar_atipicos(df_atipicos, {"Datos": None, "K-means": km_atipicos.labels_,
                              "PAM": pam_atipicos.labels_, "CLARA": clara_atipicos.labels_})

### Explorar el número de atípicos

El control regenera el ejemplo para 0–8 atípicos, manteniendo $k=2$, los algoritmos y la semilla. Sus modelos son locales: **no sobrescribe** `km_atipicos`, `pam_atipicos` ni `clara_atipicos`.

La secuencia de extracciones aleatorias depende de ese número: no es simplemente añadir filas a una muestra base idéntica. Úsalo para explorar sensibilidad, no como una prueba formal de estabilidad.

In [ ]:
def explorar_atipicos(numero):
    # Datos y ajustes locales: las variables del ejemplo fijo quedan intactas.
    tabla = generar_atipicos(nout=numero)
    km = ajustar_kmeans(tabla, k=2, n_init=1, random_state=SEED)
    med = pam(tabla, k=2)
    muestra = clara(tabla, k=2, random_state=SEED)
    # Ejes logarítmicos para ver el rango completo, no para ajustar los métodos.
    grafico = dibujar_atipicos(tabla, {"Datos": None, "K-means": km.labels_,
                                     "PAM": med.labels_, "CLARA": muestra.labels_}, mostrar=False)
    display(widgets.HTML(value=grafico))

In [ ]:
# Control opcional: conservar el valor inicial, los límites y la actualización al soltar.
control_atipicos = widgets.interactive(
    explorar_atipicos,
    numero=widgets.IntSlider(value=3, min=0, max=8, description="Atípicos", continuous_update=False),
)
display(panel_interactivo("Explora el efecto de los valores atípicos", control_atipicos))

### Lectura de 5.8 y microprácticas

**Con los tres atípicos del ejemplo fijo**, tanto k-means como PAM producen tamaños **1000 y 3**: juntan los dos grupos base y dedican un grupo a los puntos alejados. El representante del grupo grande es una media cercana a $(55.01,10.03)$ en k-means, pero una observación cercana a $(97.23,9.99)$ en PAM. Usar un medoide no evitó aquí perder la separación original.

CLARA obtiene tamaños **500 y 503**, con representantes cerca de $(10.07,10.17)$ y $(99.96,10.12)$: conserva los dos grupos base y asigna los tres atípicos al segundo. Es el resultado de sus muestras y de esta semilla, **no una garantía de robustez ni un óptimo exacto de PAM**. Los colores de los distintos paneles son etiquetas locales, no correspondencias automáticas entre grupos.

**Microprácticas opcionales, sin dependencias:**
1. Compara 0, 3 y 8 atípicos en el control: anota si los dos grupos base siguen separados.
2. Explica por qué cambiar únicamente los ejes a log no reduce el peso de los atípicos en el ajuste.
3. Distingue dos afirmaciones: «CLARA aproxima el coste PAM» y «CLARA conserva los grupos que esperaba ver».

Responde aparte; ningún modelo de estas exploraciones se reutiliza en el ejercicio 5-2.

## 5.9. Ejercicio 5-2 — Clientes de un mayorista (solución completa guiada)

**Intenta cada apartado antes de ejecutar su solución; las seis soluciones se conservan completas.** Las variables son gasto anual en frescos (`Fresh`), leche (`Milk`), comestibles (`Grocery`), congelados (`Frozen`), detergentes/papel (`Detergents_Paper`) y delicatessen (`Delicassen`).

Aquí **no hay una clase conocida**: los índices internos comparan geometría y los perfiles describen gasto, no exactitud ni tipos verdaderos de cliente. Conservaremos los importes originales para interpretar los grupos y una versión transformada para ajustar.

### 1. Lectura y comprobación

Lee el archivo de clientes desde `DATA_PATH`: sus seis columnas están **separadas por tabuladores**, aunque su extensión sea CSV. Comprueba filas, tipos y ausencias. En los datos actuales hay **440 clientes, seis variables y ningún valor ausente**.

In [ ]:
# Solución 1: lectura con el separador real del archivo, sin modificar sus datos.
mayorista = pd.read_csv(DATA_PATH / "Wholesale customers data.csv", sep="\t")
# Comprobación de estructura; los tipos y primeras filas permiten revisar la importación.
assert mayorista.shape[1] == 6 and not mayorista.isna().any().any()
display(mayorista.head())
display(mayorista.dtypes)
print(f"{len(mayorista)} clientes sin valores ausentes")

### 2. Descriptiva, logaritmo y estandarización

Las medias superan las medianas en las seis variables. Por ejemplo, `Milk` tiene media **5796.3**, mediana **3627** y máximo **73 498**: los gastos altos pueden dominar distancias en unidades originales.

La solución transforma **los datos del ajuste**, primero con `log1p` y después estandarizando cada columna:

$$
z_{ij}=\frac{\log(1+x_{ij})-\overline{\log(1+x_{\cdot j})}}{s_j},
$$

con $s_j$ calculada **después del logaritmo**. `log1p` comprime importes altos y admite ceros; el escalado equilibra la dispersión de las seis variables. Se cambia la pregunta geométrica: las diferencias proporcionales tienen más peso que antes, no se «corrigen» automáticamente los datos.

`mayorista_x` se usa para ajustar y medir distancias; `mayorista` conserva el gasto original para los perfiles. **No compares WSS entre estas dos escalas.** A diferencia de los ejes logarítmicos de 5.8, aquí sí hay transformación previa al ajuste.

In [ ]:
# Solución 2: resumir gastos originales, donde media, mediana y máximo son interpretables.
display(mayorista.describe().T[["mean", "50%", "std", "max"]].round(1))
# Transformación del ajuste: log1p primero, estandarización después; no alterar mayorista.
mayorista_x = escalar(np.log1p(mayorista))
# Verificar finitud y revisar el centrado/escalado resultante.
assert np.isfinite(mayorista_x.to_numpy()).all()
display(mayorista_x.describe().loc[["mean", "std"]].round(3))

### 3. Número de grupos: dos criterios descriptivos

Calcula WSS para $k=2,\ldots,10$ con **Lloyd y diez inicios por valor de $k$**, sobre las mismas variables logarítmicas estandarizadas. Busca si la reducción de WSS se atenúa: un codo no es una prueba estadística.

Calcula también la **silhouette media**: compara distancia al propio grupo y al grupo alternativo más próximo. Un valor mayor indica mejor cohesión/separación **para estos datos y esta distancia**, no un porcentaje de aciertos. Su máximo y el codo pueden diferir.

Después compararemos $k=5,6,7,8$ como pide el ejercicio, aunque la silhouette favorezca otro valor. No seleccionamos grupos por un p-valor inexistente.

In [ ]:
# Solución 3: candidatos comparables, con escala, algoritmo, inicios y semilla fijos.
candidatos_mayorista = {}
for k in range(2, 11):
    candidatos_mayorista[k] = ajustar_kmeans(mayorista_x, k, n_init=10,
                                            random_state=SEED, algoritmo="lloyd")
# WSS y silhouette se calculan sobre los mismos datos transformados.
resumen_mayorista = pd.DataFrame({
    "Inercia intra": {k: mod.inertia_ for k, mod in candidatos_mayorista.items()},
    "Silhouette media": {k: silhouette_score(mayorista_x, mod.labels_)
                         for k, mod in candidatos_mayorista.items()},
})
display(resumen_mayorista.round(3))
# Dos vistas complementarias: disminución de WSS y cohesión/separación media.
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
_ = resumen_mayorista["Inercia intra"].plot(ax=ejes[0], marker="o", color=COLOR_BASE)
_ = resumen_mayorista["Silhouette media"].plot(ax=ejes[1], marker="o", color=COLOR_ACENTO)
for eje, titulo in zip(ejes, ["Inercia (busca el codo)", "Silhouette (mayor = mejor)"]):
    _ = eje.set(xlabel="Número de grupos", title=titulo)
    _ = estilo_ejes(eje, cuadricula=True)
fig.tight_layout()
plt.show()

**Lectura de las dos curvas:** en el kernel actual, la mayor silhouette entre estos candidatos corresponde a **$k=2$: 0.290**; para $k=5$ es **0.197**. La WSS sigue descendiendo, de **1839.87** ($k=2$) a **1264.40** ($k=5$) y **930.50** ($k=10$). Ese descenso no basta para elegir tantos grupos: añade complejidad y no demuestra categorías reales. La curva no certifica un codo único ni contradice que estudiemos $k=5$ con fines didácticos.

### 4. Cuatro sistemas de ajuste de k-means

Para cada $k$ de **5 a 8**, compara sobre los **mismos datos transformados**:
1. Lloyd con un inicio.
2. Hartigan–Wong con un inicio.
3. Hartigan–Wong con diez inicios.
4. Ward euclídeo seguido de Hartigan–Wong, inicializado con las medias del corte del árbol.

**Dunn** divide separación mínima entre grupos por diámetro máximo dentro de un grupo; **silhouette** resume cohesión y separación medias. Un Dunn pequeño señala proximidad entre grupos respecto a su dispersión, pero puede depender mucho de unos pocos puntos. Ambos índices evalúan esta distancia y partición: **no revelan la verdad ni el acierto de clasificación**.

Los cuatro sistemas minimizan **WSS**, no Dunn ni silhouette. Un inicio puede quedar en un mínimo local; aumentar los inicios busca menor WSS, no garantiza mayor valor de los otros índices ni un óptimo global.

In [ ]:
# Solución 4: reutilizar la misma geometría euclídea logarítmica estandarizada.
arbol_mayorista = linkage(pdist(mayorista_x), method="ward")
distancias_mayorista = squareform(pdist(mayorista_x))
resultados_sistemas = {}
filas_indicadores = []
for k in range(5, 9):
    # Inicialización jerárquica: cortar Ward y calcular las medias de sus grupos.
    grupos_arbol = renumerar(cut_tree(arbol_mayorista, n_clusters=k).ravel())
    centros_arbol = np.array([mayorista_x.to_numpy()[grupos_arbol == j].mean(axis=0)
                              for j in range(k)])
    # Conservar los cuatro sistemas y sus semillas/inicios para cada k.
    sistemas = {
        "Lloyd (1)": ajustar_kmeans(mayorista_x, k, n_init=1, random_state=SEED,
                                     algoritmo="lloyd"),
        "Hartigan (1)": ajustar_kmeans(mayorista_x, k, n_init=1, random_state=SEED),
        "Hartigan (10)": ajustar_kmeans(mayorista_x, k, n_init=10, random_state=SEED),
        "Jerárquico + Hartigan": ajustar_kmeans(mayorista_x, k, init=centros_arbol,
                                                  random_state=SEED),
    }
    # Guardar las etiquetas completas; los índices evalúan, no dirigen el ajuste.
    for nombre, modelo in sistemas.items():
        etiquetas = modelo.labels_
        resultados_sistemas[(k, nombre)] = etiquetas
        filas_indicadores.append({"k": k, "Sistema": nombre,
                                  "Dunn": indice_dunn(distancias_mayorista, etiquetas),
                                  "Silhouette": silhouette_score(distancias_mayorista, etiquetas,
                                                                 metric="precomputed"),
                                  "Inercia": modelo.inertia_})
# Comparar solo k=5–8: cada índice puede recomendar un k diferente.
indicadores_mayorista = pd.DataFrame(filas_indicadores).set_index(["k", "Sistema"])
display(indicadores_mayorista.round(3))
print("Mejor k entre 5–8 para cada sistema e índice (no necesariamente iguales):")
display(pd.DataFrame({indice: indicadores_mayorista[indice].unstack("Sistema").idxmax()
                      for indice in ("Dunn", "Silhouette")}))

**Lectura de la comparación:** Dunn y silhouette discrepan en tres sistemas: Lloyd recomienda **7/5**, Hartigan (1) **6/5** y Jerárquico + Hartigan **8/7**, respectivamente; Hartigan (10) recomienda **8/8** entre los candidatos 5–8.

Con $k=5$, Hartigan (10) alcanza menor WSS (**1263.20**) que Hartigan (1) (**1265.34**), pero su silhouette es menor (**0.192 frente a 0.201**). Es evidencia de que optimizar WSS no equivale a optimizar todos los índices. No hay «ganador verdadero» deducible de esta tabla.

### 5. Similitud de las particiones (diapositiva 37)

Para **2 y 5 grupos**, compara Lloyd con un inicio y Hartigan–Wong con diez, ambos con la semilla **1234**, sin cambiar las variables transformadas.

El **ARI** compara pares de observaciones y corrige el acuerdo esperado por azar: 1 indica particiones idénticas, 0 un acuerdo aproximadamente aleatorio, y puede ser negativo. **No es el Rand sin ajustar ni exactitud.** Un acuerdo alto entre algoritmos tampoco valida la utilidad comercial de sus segmentos.

In [ ]:
# Solución 5: comparar métodos con el mismo k, escala y semilla 1234 del enunciado.
comparacion_ari = {}
for k in (2, 5):
    lloyd = ajustar_kmeans(mayorista_x, k, algoritmo="lloyd", n_init=1, random_state=1234)
    hartigan = ajustar_kmeans(mayorista_x, k, n_init=10, random_state=1234)
    # Los números de etiqueta son arbitrarios; ARI compara las relaciones entre pares.
    comparacion_ari[k] = adjusted_rand_score(lloyd.labels_, hartigan.labels_)
display(pd.Series(comparacion_ari, name="Rand ajustado: Lloyd (1) / Hartigan (10)").round(3))

### 6. Describir los grupos en gasto original

**Con la semilla 1234 del apartado 5**, el ARI es **0.991 para $k=2$** y **0.549 para $k=5$**. El acuerdo entre esos ajustes es mucho menor al distinguir cinco grupos; no son porcentajes de clientes acertados.

Para describir perfiles usamos, como **ejemplo didáctico**, Hartigan–Wong con diez inicios y **$k=5$** del apartado 4, que emplea `SEED`. No es un óptimo demostrado ni se elige por vencer en silhouette.

Muestra tamaños y **medianas en gasto original**. En los diagramas de caja representamos $\log(1+x)$, **sin estandarizar**, para hacer legibles los importes extremos; esa escala no es la de las medianas de la tabla ni permite llamar «gasto» a los centros estandarizados. Las etiquetas de grupo son arbitrarias y las diferencias son descriptivas, no causales.

In [ ]:
# Solución 6: reutilizar la partición completa k=5 de Hartigan (10), elección didáctica.
grupos_mayorista = resultados_sistemas[(5, "Hartigan (10)")]
print("Tamaño de cada grupo:")
display(pd.Series(grupos_mayorista + 1, name="Grupo").value_counts().sort_index())
# Perfiles interpretables: medianas de los importes originales, no centros estandarizados.
perfiles_mayorista = mayorista.groupby(grupos_mayorista + 1).median()
perfiles_mayorista.index.name = "Grupo"
display(perfiles_mayorista.round(0))
# Seis diagramas en log1p del gasto, sin escalado; las etiquetas no son clases conocidas.
fig, ejes = plt.subplots(2, 3, figsize=(14, 8))
for eje, nombre in zip(ejes.flat, mayorista.columns):
    cajas = [np.log1p(mayorista.loc[grupos_mayorista == g, nombre]) for g in range(5)]
    grafico = eje.boxplot(cajas, patch_artist=True,
                          medianprops={"color": "#273443", "linewidth": 1.8},
                          flierprops={"marker": ".", "markersize": 3, "alpha": 0.35})
    for g, caja in enumerate(grafico["boxes"]):
        _ = caja.set(facecolor=COLORES_GRUPO[g], edgecolor=COLORES_GRUPO[g], alpha=0.5)
    _ = eje.set(title=nombre, xlabel="Grupo", ylabel="log(1 + gasto)")
    _ = estilo_ejes(eje, cuadricula=True)
fig.tight_layout()
plt.show()

### Lectura de los perfiles y microprácticas de 5.9

En la partición didáctica, los grupos **1–5** tienen **100, 100, 98, 54 y 88 clientes**. Sus medianas originales permiten describir contrastes; las cifras citadas usan `perfiles_mayorista` sin redondear, mientras la tabla mostrada redondea a enteros:
- El grupo **1** presenta las mayores medianas de `Milk` (**9769.5**), `Grocery` (**13 576.5**) y `Detergents_Paper` (**5495**).
- El grupo **2** destaca en `Fresh` (**18 429**) y `Frozen` (**5857.5**).
- El grupo **4** combina `Grocery` alto (**11 207**) con `Fresh` bajo (**992**); el **5** tiene las menores medianas de `Milk`, `Grocery` y `Detergents_Paper`.

El grupo 3 y el resto de variables se leen en la tabla y las seis cajas. El solapamiento y los puntos extremos siguen importando: estos perfiles **no identifican por sí solos restaurantes, comercios u otras clases de cliente**.

**Microprácticas opcionales, agrupadas y sin dependencias:**
1. Justifica por escrito qué aporta estudiar $k=5$ si la silhouette de los candidatos favorece $k=2$.
2. Señala un caso donde menor WSS no implique mayor silhouette y otro donde Dunn y silhouette recomienden distinto $k$.
3. Describe dos grupos citando medianas originales y tamaños; separa esa lectura de la escala logarítmica de las cajas.
4. Explica por qué ARI = 0.991 no equivale a 99.1 % de exactitud comercial.

Las seis soluciones anteriores son completas: ninguna necesita respuestas, variables ni modificaciones de estas prácticas.

## Cierre de la sesión

Las figuras realmente obtenidas ilustran decisiones distintas:
- **PAM en Iris:** la proyección acompaña un ARI = 1 frente a k-means, no una exactitud del 100 % ni una igualdad de costes.
- **Disco y corona:** las nubes y el panel del kernel muestran una separación radial con $\gamma=2$ que k-means euclídeo no recupera.
- **Cuatro paneles con atípicos:** k-means y PAM aíslan los tres puntos extremos; CLARA conserva los dos grupos base en esta ejecución. Los ejes log solo hacen legible el dibujo.
- **Mayorista:** las curvas de WSS/silhouette favorecen distintos compromisos; la silhouette máxima está en $k=2$. Las seis cajas y las medianas originales describen el $k=5$ didáctico, no un óptimo probado.

**Conclusión breve:** cita una figura y una cifra para justificar una decisión, sin convertir etiquetas en clases conocidas. Escala, geometría, inicialización y muestreo condicionan el resultado; ARI, Dunn y silhouette no demuestran categorías naturales, utilidad comercial ni causalidad. La estabilidad y la utilidad fuera de esta muestra siguen sin validarse. Todas las microprácticas son opcionales y no alimentan los ejemplos.

### Documentación

Referencias para interpretar los algoritmos y los índices; no añaden dependencias al cuaderno:
- [Clustering jerárquico y Ward en SciPy](https://docs.scipy.org/doc/scipy/reference/cluster.hierarchy.html).
- [K-means en scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html).
- [PAM: referencia algorítmica en el manual de cluster](https://stat.ethz.ch/R-manual/R-devel/library/cluster/html/pam.html) y [CLARA](https://stat.ethz.ch/R-manual/R-devel/library/cluster/html/clara.html); aquí se usan las implementaciones Python del cuaderno.
- [Kernel gaussiano RBF: similitud, gamma y distancia](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.rbf_kernel.html).
- [Índice de Rand ajustado (ARI)](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html).
- [Silhouette: cohesión y separación](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html).
- [Selección del número de grupos mediante indicadores internos](https://www.jstatsoft.org/article/view/v061i06).

In [ ]:
# VALIDACION_S5_TEMPORAL: referencia previa a la extracción; se retirará al finalizar.
def _referencia_s5():
    import ast
    import json
    from pathlib import Path
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    from ipywidgets import Widget

    fichero = Path('/Users/I0551731/Desktop/LaSalle_Estadistica/sesiones/sesion5/s5.ipynb')
    contenido = json.loads(fichero.read_text(encoding='utf-8'))
    shell = get_ipython()
    contexto = {'__name__': '__main__', 'get_ipython': get_ipython}
    fuentes = []
    publicar = shell.display_pub.publish
    figuras_previas = set(plt.get_fignums())
    try:
        shell.display_pub.publish = lambda *args, **kwargs: None
        for numero, celda in enumerate(contenido['cells'], 1):
            fuente = celda['source']
            fuente = fuente if isinstance(fuente, str) else ''.join(fuente)
            if fuente.startswith('# VALIDACION_S5_TEMPORAL'):
                continue
            fuentes.append((celda['cell_type'], fuente))
            if celda['cell_type'] != 'code':
                continue
            arbol = ast.parse(shell.input_transformer_manager.transform_cell(fuente))
            with capture_output(display=False):
                exec(compile(arbol, f'<S5 anterior: celda {numero}>', 'exec'), contexto)
    finally:
        shell.display_pub.publish = publicar
        vistos = set()
        def cerrar(objeto):
            if not isinstance(objeto, Widget) or id(objeto) in vistos:
                return
            vistos.add(id(objeto))
            for hijo in getattr(objeto, 'children', ()):
                cerrar(hijo)
            objeto.close()
        for objeto in contexto.values():
            cerrar(objeto)
        for numero in set(plt.get_fignums()) - figuras_previas:
            plt.close(numero)
    lineas = sum(len(f.splitlines()) for tipo, f in fuentes if tipo == 'code')
    print(f'Referencia S5: {sum(tipo == "code" for tipo, _ in fuentes)} celdas y {lineas} líneas de código.')
    print(f'WSS de k-means Iris: {contexto["km3"].inertia_:.8f}; Hopkins: {contexto["hop"]:.8f}.')
    print(f'Sistemas del mayorista conservados: {len(contexto["resultados_sistemas"])}.')
    return {'fuentes': fuentes, 'contexto': contexto, 'lineas': lineas}

_s5_referencia = _referencia_s5()
del _referencia_s5